# Data Scraping & Pipeline — End-to-End Notebook

**RAG-Based LLM Code Review Agent** | IIT Madras DSAI Lab — Group 1

This single notebook handles the complete data pipeline:

1. **GitHub Scraping** — Fetch PR review comments from 5 Python repos via PyGithub
2. **LLM Classification** — Multi-model batch classification of violations (GPT-4.1 / GPT-4o / GPT-4.1-mini)
3. **PEP 8/257 Scraping** — Best-practice knowledge from official PEP documents
4. **Contributing Guidelines** — Repo-specific style guidelines
5. **Linter Rules** — Curated Flake8, Pylint, pycodestyle, pydocstyle rules
6. **Retrieval Corpus Assembly** — Deduplicated, chunked knowledge base
7. **Evaluation Dataset Construction** — Ground-truth entries from flask & fastapi
8. **Static Analysis Input** — Full source files with modified-line tracking
9. **Ground-Truth Cross-Validation** — Independent LLM re-classification
10. **Template-Based Synthetic Augmentation** — 10+ realistic code templates per category
11. **LLM Synthetic Augmentation** — Dynamic code snippet & corpus generation
12. **Data Cleanup & Validation** — 15-point quality scoring
13. **EDA Visualizations** — 5 publication-quality figures
14. **Train / Validation / Test Split** — Repository-level separation with leakage checks



In [ ]:
%pip install PyGithub openai beautifulsoup4 lxml pandas tqdm requests ipywidgets matplotlib seaborn numpy --quiet

In [ ]:
import os
import re
import json
import time
import hashlib
import logging
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from typing import Optional

import requests as req
import pandas as pd
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from github import Github, Auth, GithubException, RateLimitExceededException
from openai import OpenAI

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-7s │ %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("datascrape")
log.info("All imports successful ")

In [ ]:
GITHUB_TOKEN = "xxxxxxx"

gh = Github(auth=Auth.Token(GITHUB_TOKEN), per_page=100)
try:
    rl = gh.get_rate_limit()
    rate = rl.rate
    log.info(f"GitHub authenticated — Rate limit: {rate.remaining}/{rate.limit} (resets {rate.reset})")
except GithubException as e:
    raise SystemExit(f"GitHub auth failed: {e}")

MODEL_POOL = ["gpt-4.1", "gpt-4o", "gpt-4.1-mini"]

llm_clients = {}
available_models = []

for model_name in MODEL_POOL:
    client = OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKEN,
    )
    try:
        _test = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=5,
        )
        llm_clients[model_name] = client
        available_models.append(model_name)
        log.info(f"{model_name} authenticated — Response: {_test.choices[0].message.content}")
    except Exception as e:
        log.warning(f"{model_name} unavailable: {e}")

LLM_AVAILABLE = len(available_models) > 0
llm_client = llm_clients.get(available_models[0]) if available_models else None
LLM_MODEL = available_models[0] if available_models else "gpt-4.1"

log.info(f"Models available: {len(available_models)}/{len(MODEL_POOL)} — {available_models}")
log.info(f"Effective RPM: ~{len(available_models) * 15} (load balanced)")


In [ ]:
TRAIN_REPOS = ["django/django", "pandas-dev/pandas", "scikit-learn/scikit-learn"]
EVAL_REPOS  = ["pallets/flask", "fastapi/fastapi"]
ALL_REPOS   = TRAIN_REPOS + EVAL_REPOS

CATEGORIES = [
    "indentation",
    "naming_convention",
    "unused_import",
    "mutable_default",
    "documentation_formatting",
]

TARGET_PER_CATEGORY_PER_REPO = 1000
MIN_VIOLATIONS_PER_REPO = 50
SCRAPE_COMMENTS_PER_REPO = 5000        # scrape more to get enough per category

GH_PER_PAGE = 100
GH_RATE_LIMIT_BUFFER = 100

LLM_RPM_LIMIT = 15                    # per-model rate limit
LLM_RETRY_MAX = 5
LLM_RETRY_BASE_DELAY = 2
LLM_BATCH_SIZE = 80                   # comments per API call (aggressive batching)

CHUNK_MIN_TOKENS = 100
CHUNK_MAX_TOKENS = 400

OUTPUT_DIR = Path(".")
CACHE_DIR  = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

EVAL_DATASET_PATH       = OUTPUT_DIR / "evaluation_dataset.json"
RETRIEVAL_CORPUS_PATH   = OUTPUT_DIR / "retrieval_corpus.json"
STATIC_ANALYSIS_PATH    = OUTPUT_DIR / "static_analysis_input.json"

log.info(f"Config loaded ")
log.info(f" Train repos : {TRAIN_REPOS}")
log.info(f" Eval repos : {EVAL_REPOS}")
log.info(f" Categories : {CATEGORIES}")
log.info(f" Target/repo : {MIN_VIOLATIONS_PER_REPO} ({TARGET_PER_CATEGORY_PER_REPO}/category)")
log.info(f" Models : {available_models} (load balanced)")
log.info(f" Batch size : {LLM_BATCH_SIZE} comments/call")

In [ ]:
class GitHubRateLimiter:
    """Monitors GitHub rate limits and sleeps when necessary."""

    def __init__(self, github_client: Github, buffer: int = GH_RATE_LIMIT_BUFFER):
        self.gh = github_client
        self.buffer = buffer
        self.calls_made = 0

    def check_and_wait(self):
        """Check remaining rate limit; sleep if below buffer."""
        self.calls_made += 1
        if self.calls_made % 50 == 0:  # check every 50 calls to avoid wasting requests
            rate = self.gh.get_rate_limit().rate
            remaining = rate.remaining
            if remaining < self.buffer:
                reset_time = rate.reset.replace(tzinfo=timezone.utc)
                now = datetime.now(timezone.utc)
                wait_seconds = max((reset_time - now).total_seconds() + 5, 1)
                log.warning(
                    f"⏳ Rate limit low ({remaining} remaining). "
                    f"Sleeping {wait_seconds:.0f}s until {rate.reset}"
                )
                time.sleep(wait_seconds)
                log.info(" Rate limit reset — resuming")

    def safe_call(self, func, *args, max_retries=5, **kwargs):
        """Call a GitHub API function with retry + backoff on rate limit errors."""
        for attempt in range(max_retries):
            self.check_and_wait()
            try:
                return func(*args, **kwargs)
            except RateLimitExceededException:
                rate = self.gh.get_rate_limit().rate
                reset_time = rate.reset.replace(tzinfo=timezone.utc)
                now = datetime.now(timezone.utc)
                wait = max((reset_time - now).total_seconds() + 5, 30)
                log.warning(f" Rate limit exceeded. Sleeping {wait:.0f}s (attempt {attempt+1})")
                time.sleep(wait)
            except GithubException as e:
                if e.status in (403, 429, 502, 503):
                    delay = min(2 ** attempt * 5, 120)
                    log.warning(f"️ GitHub error {e.status}. Retrying in {delay}s (attempt {attempt+1})")
                    time.sleep(delay)
                else:
                    raise
        raise RuntimeError(f"GitHub API call failed after {max_retries} retries")


rate_limiter = GitHubRateLimiter(gh)


def save_cache(data, name: str):
    """Save data to a JSON cache file."""
    path = CACHE_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
        log.info(f" Cached {name} → {path} ({len(data)} items)")

def load_cache(name: str):
    """Load data from a JSON cache file if it exists."""
    path = CACHE_DIR / f"{name}.json"
    if path.exists():
        with open(path, "r") as f:
            data = json.load(f)
            log.info(f" Loaded cache {name} ← {path} ({len(data)} items)")
        return data
    return None

def parse_diff_hunks(patch: str) -> list[dict]:
    """Parse a unified diff patch into structured hunks."""
    if not patch:
        return []
    hunks = []
    current_hunk = None
    hunk_header_re = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")

    for line in patch.split("\n"):
        m = hunk_header_re.match(line)
        if m:
            if current_hunk:
                hunks.append(current_hunk)
            current_hunk = {
                "old_start": int(m.group(1)),
                "new_start": int(m.group(2)),
                "lines": [],
            }
        elif current_hunk is not None:
            current_hunk["lines"].append(line)

    if current_hunk:
        hunks.append(current_hunk)

    for h in hunks:
        new_line = h["new_start"]
        for l in h["lines"]:
            if l.startswith("+") or (not l.startswith("-") and l != "\\ No newline at end of file"):
                new_line += 1
        h["new_end"] = max(new_line - 1, h["new_start"])

    return hunks

def estimate_tokens(text: str) -> int:
    """Rough token count (words × 1.3)."""
    return int(len(text.split()) * 1.3)

log.info("Utilities loaded ")

In [ ]:
def fetch_review_comments(repo_name: str, max_comments: int = SCRAPE_COMMENTS_PER_REPO) -> list[dict]:
    """
    Fetch inline review comments from a repo.
    Uses the repo-wide endpoint (efficient: no need to iterate per-PR).
    Filters to Python files only.
    IMPORTANT: Avoids comment.raw_data access (triggers extra API call per comment).
    """
    cache_key = f"raw_comments_{repo_name.replace('/', '_')}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached

    log.info(f" Scraping review comments from {repo_name} (max {max_comments})...")
    repo = rate_limiter.safe_call(gh.get_repo, repo_name)

    comments = []
    try:
        paginated = repo.get_pulls_review_comments(sort="created", direction="desc")

        for comment in tqdm(paginated, desc=f"{repo_name} comments", total=min(max_comments, paginated.totalCount)):
            rate_limiter.check_and_wait()

            if not comment.path.endswith(".py"):
                continue

            if comment.user and comment.user.type == "Bot":
                continue

            if not comment.body or len(comment.body.strip()) < 15:
                continue

            pr_number = int(comment.pull_request_url.rstrip("/").split("/")[-1])

            raw = comment._rawData if hasattr(comment, '_rawData') else {}

            comments.append({
                "comment_id": comment.id,
                "repo": repo_name,
                "pr_number": pr_number,
                "path": comment.path,
                "line": comment.line,
                "original_line": comment.original_line,
                "start_line": comment.start_line,
                "side": comment.side if hasattr(comment, "side") else None,
                "diff_hunk": comment.diff_hunk,
                "body": comment.body,
                "author_association": raw.get("author_association", "NONE"),
                "user_login": comment.user.login if comment.user else None,
                "created_at": str(comment.created_at),
                "in_reply_to_id": raw.get("in_reply_to_id"),
            })

            if len(comments) >= max_comments:
                break

    except Exception as e:
        log.error(f"Error scraping {repo_name}: {e}")
        import traceback; traceback.print_exc()

    log.info(f" Scraped {len(comments)} Python review comments from {repo_name}")
    save_cache(comments, cache_key)
    return comments


def fetch_pr_files(repo_name: str, pr_number: int) -> list[dict]:
    """Fetch all file diffs for a specific PR."""
    cache_key = f"pr_files_{repo_name.replace('/', '_')}_{pr_number}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached

    repo = rate_limiter.safe_call(gh.get_repo, repo_name)
    pr = rate_limiter.safe_call(repo.get_pull, pr_number)

    files = []
    for f in pr.get_files():
        rate_limiter.check_and_wait()
        if not f.filename.endswith(".py"):
            continue
        files.append({
            "filename": f.filename,
            "status": f.status,
            "additions": f.additions,
            "deletions": f.deletions,
            "changes": f.changes,
            "patch": f.patch if f.patch else "",
        })

    save_cache(files, cache_key)
    return files


def fetch_contributing_guide(repo_name: str) -> Optional[str]:
    """Fetch CONTRIBUTING.md or similar style guide from a repo."""
    cache_key = f"contributing_{repo_name.replace('/', '_')}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached[0] if cached else None

    repo = rate_limiter.safe_call(gh.get_repo, repo_name)

    guide_files = [
        "CONTRIBUTING.md",
        "CONTRIBUTING.rst",
        ".github/CONTRIBUTING.md",
        "docs/contributing.md",
        "docs/contributing.rst",
    ]

    content = None
    for fname in guide_files:
        try:
            file_content = rate_limiter.safe_call(repo.get_contents, fname)
            if file_content and hasattr(file_content, "decoded_content"):
                content = file_content.decoded_content.decode("utf-8", errors="replace")
                log.info(f" Found {fname} in {repo_name} ({len(content)} chars)")
                break
        except GithubException:
            continue

    save_cache([content] if content else [], cache_key)
    return content

log.info("GitHub scraping functions defined ")

In [ ]:
BATCH_SYSTEM_PROMPT = """You are a STRICT Python code review classifier. You will receive a JSON array of review comments from GitHub pull requests. Your job is to classify EACH comment into EXACTLY ONE of these 6 labels:

ALLOWED CATEGORIES (use ONLY if the comment is CLEARLY and PRIMARILY about this topic):
- indentation: The comment explicitly discusses indent levels, tabs vs spaces, alignment of continuation lines, wrong indentation depth, or whitespace structure of code blocks. NOT about line length, NOT about general formatting.
- naming_convention: The comment explicitly discusses the name of a variable, function, class, or module — e.g., "should use snake_case", "rename this to X", "name is misleading", PEP8 naming style.
- unused_import: The comment explicitly mentions an import that is unused, redundant, should be removed, or was imported but never referenced.
- mutable_default: The comment explicitly mentions a mutable default argument (list, dict, set as default parameter value), the None sentinel pattern, or warns about shared mutable defaults.
- documentation_formatting: The comment explicitly discusses docstrings, documentation formatting, missing documentation, doc style (numpy/google/sphinx), or documentation-related changes.
- none: EVERYTHING ELSE. This is the DEFAULT. Use it for any comment that does not CLEARLY fit one of the 5 categories above.

MUST classify as "none" (these are VERY COMMON traps):
- Bug reports, logic errors, off-by-one errors → none
- Performance suggestions → none
- API design discussions, interface changes → none
- "Add type hints" without docstring context → none
- Code simplification, refactoring suggestions → none
- Test-related comments → none
- Compatibility or deprecation warnings → none
- Line length / max line length discussions → none
- Security concerns → none
- Error handling / exception suggestions → none
- General "clean up" or "style" without specific category → none
- Comments about function arguments (not about their names or defaults) → none
- Comments about return values, types, or control flow → none

CRITICAL RULES:
1. When in doubt, classify as "none". False negatives are MUCH better than false positives.
2. Read the ACTUAL CONTENT of the comment, not just keywords. A comment mentioning "import" in passing is NOT about unused imports.
3. A comment about "indentation" must be about ACTUAL indent levels/whitespace, not about code organization or structure.
4. A comment that says "this default argument" is not mutable_default unless it specifically discusses mutability (list/dict/set defaults).

Respond with ONLY a JSON array of category strings in the SAME ORDER as the input.
Example input: [{"id":0,"body":"fix indent level to 4 spaces"},{"id":1,"body":"unused os import"},{"id":2,"body":"this will break on Python 3.12"}]
Example output: ["indentation","unused_import","none"]
No markdown, no explanation — just the raw JSON array."""

_CATEGORY_PATTERNS: dict[str, list[str]] = {
    "indentation": [
        r"\bindent(ation|ed)?\b", r"\bdedent", r"\btab[s]?\b",
        r"\bwhitespace\b", r"mixed\s*tabs?\s*(and|&)\s*spaces?",
        r"\balign(ment|ed)?\b", r"continuation\s*line",
        r"\bE1[0-9]{2}\b", r"\bW1[0-9]{2}\b",
        r"(wrong|incorrect|bad|fix|extra|missing)\s*indent",
        r"(over|under)\s*-?\s*indent", r"reindent",
    ],
    "naming_convention": [
        r"\bnaming\b", r"\bsnake[_\s]?case\b", r"\bcamel[_\s]?case\b",
        r"\brename\s+(this|it|the|to)\b",
        r"(variable|function|method|class|module|constant)\s*name",
        r"\bN[0-9]{3}\b", r"\bE741\b",
        r"should\s*be\s*named", r"doesn.t\s*follow.*naming",
        r"(descriptive|meaningful|better)\s*name",
    ],
    "unused_import": [
        r"unused\s*import", r"import.*unused",
        r"imported\s*but\s*(not|never)\s*used",
        r"remove.*import", r"unnecessary\s*import",
        r"\bF401\b", r"\bW0611\b",
        r"import.*not\s*(needed|required|necessary)",
        r"dead\s*import", r"leftover\s*import",
    ],
    "mutable_default": [
        r"mutable\s*default", r"default\s*argument.*mutable",
        r"(def|function).*=\s*\[\]", r"(def|function).*=\s*\{\}",
        r"\bB006\b", r"\bB008\b", r"\bW0102\b",
        r"dangerous\s*default", r"default.*empty\s*(list|dict|set)",
        r"use\s*None.*default", r"default\s*arg.*mutable",
    ],
    "documentation_formatting": [
        r"\bdocstring", r"missing\s*doc(umentation|string)?",
        r"add\s*(a\s*)?doc(string)?",
        r"(numpy|google|sphinx|rst)\s*(style|format|docstring)",
        r"\bD[1234][0-9]{2}\b",
        r"docstring\s*(format|style|convention|missing|should|needs|add)",
        r"documentation\s*(format|style|missing|should|needs|update)",
        r"missing\s*(param|return|type|description).*doc",
        r"triple\s*quote",
    ],
}

_COMPILED_PATTERNS: dict[str, re.Pattern] = {
    cat: re.compile("|".join(patterns), re.IGNORECASE)
    for cat, patterns in _CATEGORY_PATTERNS.items()
}

VALID_LABELS = set(CATEGORIES) | {"none"}


def classify_comment_keyword(comment_body: str, diff_hunk: str = "") -> str:
    """Classify using keyword/regex matching (fallback)."""
    text = f"{comment_body} {diff_hunk}".lower()
    scores: dict[str, int] = {}
    for cat, pattern in _COMPILED_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            scores[cat] = len(matches)
    if not scores:
        return "none"
    return max(scores, key=scores.get)


def _build_batch_items(comments: list[dict]) -> list[dict]:
    """Build compact JSON items for a batch LLM call."""
    items = []
    for i, c in enumerate(comments):
        item = {"id": i, "body": c["body"][:500]}
        hunk = c.get("diff_hunk", "")
        if hunk:
            item["code"] = hunk[:400]
        items.append(item)
    return items


_model_call_index = 0
_model_last_call_time = {m: 0.0 for m in available_models}


def _get_next_model():
    """Round-robin model selection with per-model rate limiting."""
    global _model_call_index
    if not available_models:
        return None, None
    
    now = time.time()
    for offset in range(len(available_models)):
        idx = (_model_call_index + offset) % len(available_models)
        model_name = available_models[idx]
        last_call = _model_last_call_time[model_name]
        min_interval = 60.0 / LLM_RPM_LIMIT
        
        if now - last_call >= min_interval:
            _model_call_index = (idx + 1) % len(available_models)
            _model_last_call_time[model_name] = now
            return model_name, llm_clients[model_name]
    
    best_model = min(available_models, key=lambda m: _model_last_call_time[m])
    wait = (60.0 / LLM_RPM_LIMIT) - (now - _model_last_call_time[best_model])
    if wait > 0:
        time.sleep(wait)
    _model_last_call_time[best_model] = time.time()
    idx = available_models.index(best_model)
    _model_call_index = (idx + 1) % len(available_models)
    return best_model, llm_clients[best_model]


def classify_batch_llm(comments: list[dict]) -> list[str]:
    """
    Classify a batch of comments in ONE LLM call.
    Uses round-robin model selection for load balancing.
    """
    batch_items = _build_batch_items(comments)
    user_msg = json.dumps(batch_items, ensure_ascii=False)

    for attempt in range(LLM_RETRY_MAX):
        model_name, client = _get_next_model()
        if not client:
            break
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": BATCH_SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=len(comments) * 30,
                temperature=0.0,
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
            labels = json.loads(raw)
            if isinstance(labels, list) and len(labels) == len(comments):
                return [
                    lbl.strip().lower() if isinstance(lbl, str) and lbl.strip().lower() in VALID_LABELS else "none"
                    for lbl in labels
                ]
                log.warning(f"[{model_name}] Length mismatch: got {len(labels)}, expected {len(comments)}")
        except json.JSONDecodeError:
            log.warning(f"[{model_name}] JSON parse error (attempt {attempt+1})")
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "rate" in err_str or "quota" in err_str:
                delay = LLM_RETRY_BASE_DELAY * (2 ** attempt)
                log.warning(f" [{model_name}] rate limit (attempt {attempt+1}). Sleeping {delay}s...")
                time.sleep(delay)
            else:
                log.warning(f"[{model_name}] error: {e}")
                break
            log.warning(f"Falling back to keyword classifier for batch of {len(comments)}")
    return [classify_comment_keyword(c["body"], c.get("diff_hunk", "")) for c in comments]


def classify_comments_batch(comments: list[dict], repo_name: str) -> list[dict]:
    """
    Classify all comments for a repo using batched, load-balanced LLM calls.
    """
    cache_key = f"classified_{repo_name.replace('/', '_')}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached

    n_models = len(available_models)
    method = f"LOAD-BALANCED ({n_models} models, {LLM_BATCH_SIZE}/call)" if LLM_AVAILABLE else "keyword-based"
    log.info(f" Classifying {len(comments)} comments from {repo_name} [{method}]...")

    classified = []
    n_batches = (len(comments) + LLM_BATCH_SIZE - 1) // LLM_BATCH_SIZE

    for batch_idx in tqdm(range(n_batches), desc=f"Classifying {repo_name}"):
        start = batch_idx * LLM_BATCH_SIZE
        end = min(start + LLM_BATCH_SIZE, len(comments))
        batch = comments[start:end]

        if LLM_AVAILABLE:
            labels = classify_batch_llm(batch)
        else:
            labels = [classify_comment_keyword(c["body"], c.get("diff_hunk", "")) for c in batch]

        for comment, label in zip(batch, labels):
            classified.append({**comment, "violation_category": label})

        cats = Counter(c["violation_category"] for c in classified if c["violation_category"] != "none")
        log.info(f" Batch {batch_idx+1}/{n_batches} — {len(classified)}/{len(comments)} — Relevant: {sum(cats.values())}")

    save_cache(classified, cache_key)

    cats = Counter(c["violation_category"] for c in classified)
    relevant = {k: v for k, v in cats.items() if k != "none"}
    log.info(f" {repo_name} complete — Relevant: {sum(relevant.values())} | None: {cats.get('none', 0)}")
    for cat, count in sorted(relevant.items()):
        log.info(f" {cat}: {count}")

    return classified


log.info(f"Classification engine ready — Mode: {'LOAD-BALANCED (' + str(len(available_models)) + ' models)' if LLM_AVAILABLE else 'keyword fallback'}")

In [ ]:
all_raw_comments = {}
for repo_name in ALL_REPOS:
    log.info(f"\n{'='*60}")
    log.info(f"Processing: {repo_name}")
    log.info(f"{'='*60}")
    all_raw_comments[repo_name] = fetch_review_comments(repo_name)

total = sum(len(v) for v in all_raw_comments.values())
log.info(f"\n Total raw comments scraped: {total}")
for repo_name, comments in all_raw_comments.items():
    log.info(f" {repo_name}: {len(comments)}")

In [ ]:
all_classified = {}
for repo_name in ALL_REPOS:
    log.info(f"\n{'='*60}")
    log.info(f"Classifying: {repo_name}")
    log.info(f"{'='*60}")
    all_classified[repo_name] = classify_comments_batch(
        all_raw_comments[repo_name], repo_name
    )

log.info(f"\n{'='*60}")
log.info(" Classification Summary")
log.info(f"{'='*60}")

summary_rows = []
for repo_name, classified in all_classified.items():
    cats = Counter(c["violation_category"] for c in classified if c["violation_category"] != "none")
    row = {"repo": repo_name, "total_relevant": sum(cats.values()), **{c: cats.get(c, 0) for c in CATEGORIES}}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
import random
random.seed(42)

def stratified_sample(classified_comments: list[dict], repo_name: str, target_per_cat: int = TARGET_PER_CATEGORY_PER_REPO) -> list[dict]:
    """
    Sample up to target_per_cat comments per violation category.
    Prioritizes maintainer comments (OWNER, MEMBER, COLLABORATOR).
    """
    by_category = defaultdict(list)
    for c in classified_comments:
        cat = c["violation_category"]
        if cat != "none" and cat in CATEGORIES:
            by_category[cat].append(c)

    sampled = []
    for cat in CATEGORIES:
        pool = by_category[cat]
        if not pool:
            log.warning(f" ️ {repo_name}: No comments for {cat}")
            continue

        maintainer = [c for c in pool if c.get("author_association") in ("OWNER", "MEMBER", "COLLABORATOR")]
        contributor = [c for c in pool if c.get("author_association") == "CONTRIBUTOR"]
        others = [c for c in pool if c.get("author_association") not in ("OWNER", "MEMBER", "COLLABORATOR", "CONTRIBUTOR")]

        prioritized = maintainer + contributor + others

        seen = set()
        unique = []
        for c in prioritized:
            key = f"{c['path']}:{c['line']}:{c['body'][:80]}"
            if key not in seen:
                seen.add(key)
                unique.append(c)

        n = min(target_per_cat, len(unique))
        sampled.extend(unique[:n])
        log.info(f" {repo_name} | {cat}: {n}/{len(pool)} sampled (from {len(unique)} unique)")

    return sampled


all_sampled = {}
for repo_name, classified in all_classified.items():
    log.info(f"\nSampling {repo_name}:")
    all_sampled[repo_name] = stratified_sample(classified, repo_name)

log.info(f"\n{'='*60}")
log.info(" Stratified Sampling Summary")
log.info(f"{'='*60}")
for repo_name, sampled in all_sampled.items():
    cats = Counter(c["violation_category"] for c in sampled)
    log.info(f" {repo_name}: {len(sampled)} total — {dict(cats)}")

In [ ]:
all_pr_files = {}  # (repo, pr_number) -> [file dicts]

for repo_name in ALL_REPOS:
    sampled = all_sampled[repo_name]
    unique_prs = sorted(set(c["pr_number"] for c in sampled))
    log.info(f"\n Fetching diffs for {len(unique_prs)} PRs from {repo_name}...")

    for pr_num in tqdm(unique_prs, desc=f"{repo_name} PR diffs"):
        key = (repo_name, pr_num)
        if key not in all_pr_files:
            try:
                files = fetch_pr_files(repo_name, pr_num)
                all_pr_files[key] = files
            except Exception as e:
                log.warning(f" ️ Failed to fetch PR #{pr_num} from {repo_name}: {e}")
                all_pr_files[key] = []

log.info(f"\n Fetched diffs for {len(all_pr_files)} PRs total")

In [ ]:
PEP8_SECTION_MAP = {
    "indentation": "indentation",
    "tabs or spaces": "indentation",
    "maximum line length": "indentation",
    "should a line break before or after a binary operator": "indentation",
    "blank lines": "indentation",
    "naming conventions": "naming_convention",
    "overriding principle": "naming_convention",
    "descriptive: naming styles": "naming_convention",
    "prescriptive: naming conventions": "naming_convention",
    "package and module names": "naming_convention",
    "class names": "naming_convention",
    "type variable names": "naming_convention",
    "exception names": "naming_convention",
    "global variable names": "naming_convention",
    "function and variable names": "naming_convention",
    "function and method arguments": "naming_convention",
    "method names and instance variables": "naming_convention",
    "constants": "naming_convention",
    "names to avoid": "naming_convention",
    "imports": "unused_import",
    "programming recommendations": "general",
    "other recommendations": "general",
    "comments": "documentation_formatting",
    "documentation strings": "documentation_formatting",
    "block comments": "documentation_formatting",
    "inline comments": "documentation_formatting",
    "string quotes": "documentation_formatting",
}

PEP257_SECTION_MAP = {
    "what is a docstring": "documentation_formatting",
    "one-line docstrings": "documentation_formatting",
    "multi-line docstrings": "documentation_formatting",
    "handling docstring indentation": "documentation_formatting",
}


def _clean_code_tokenization(text: str) -> str:
    """
    Fix bad tokenization artifacts from BeautifulSoup's get_text(separator=' ').
    Repairs patterns like 'func ( arg , arg )' → 'func(arg, arg)'
    and 'foo [ 0 ]' → 'foo[0]', 'x = { }' → 'x = {}', etc.
    """
    import re as _re
    text = _re.sub(r'\s+\(', '(', text)
    text = _re.sub(r'\s+\)', ')', text)
    text = _re.sub(r'\s+\]', ']', text)
    text = _re.sub(r'\(\s+', '(', text)
    text = _re.sub(r'\[\s+', '[', text)
    text = _re.sub(r'\(\s*\)', '()', text)
    text = _re.sub(r'\[\s*\]', '[]', text)
    text = _re.sub(r'\{\s*\}', '{}', text)
    text = _re.sub(r'\s*,\s+', ', ', text)
    text = _re.sub(r'\s*:\s+', ': ', text)
    text = _re.sub(r'\s*\.\s*', '.', text)
    text = _re.sub(r'@\s+', '@', text)
    text = _re.sub(r'\s*=\s*', ' = ', text)
    text = _re.sub(r' = =', ' ==', text)
    text = _re.sub(r'! =', '!=', text)
    text = _re.sub(r'< =', '<=', text)
    text = _re.sub(r'> =', '>=', text)
    text = _re.sub(r'\+ =', '+=', text)
    text = _re.sub(r'- =', '-=', text)
    text = _re.sub(r'\* =', '*=', text)
    text = _re.sub(r'[^\S\n]+', ' ', text)
    return text


def _extract_section_text(heading, section_map: dict) -> tuple[str, str, str]:
    """Extract text from a section, preserving code block formatting."""
    title = heading.get_text(strip=True).lower()
    
    category = "general"
    for key, cat in section_map.items():
        if key in title:
            category = cat
            break
    
    parts = []
    sibling = heading.find_next_sibling()
    while sibling:
        if sibling.name in ["h1", "h2", "h3", "h4"]:
            break
        
        pre_blocks = sibling.find_all("pre")
        if pre_blocks:
            for pre in pre_blocks:
                pre.extract()  # temporarily remove
            non_code_text = sibling.get_text(separator=" ", strip=True)
            if non_code_text:
                parts.append(non_code_text)
            for pre in pre_blocks:
                code_text = pre.get_text()  # NO separator — preserve original spacing
                if code_text.strip():
                    parts.append(f"\n```python\n{code_text.strip()}\n```\n")
        else:
            text = sibling.get_text(separator=" ", strip=True)
            if text:
                parts.append(text)
        
        sibling = sibling.find_next_sibling()
    
    full_text = f"{heading.get_text(strip=True)}\n\n" + "\n\n".join(parts)
    return full_text, category, title


def scrape_pep(url: str, section_map: dict, source_type: str) -> list[dict]:
    """
    Scrape a PEP page, split into sections, chunk text, and map to categories.
    Preserves code block formatting (no bad tokenization).
    """
    cache_key = f"pep_{source_type}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached

    log.info(f" Scraping {url}...")
    resp = req.get(url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    content = soup.find("section") or soup.find("article") or soup.find("div", class_="body")
    if not content:
        content = soup.find("main") or soup

    chunks = []
    chunk_counter = 0

    headings = content.find_all(["h1", "h2", "h3", "h4"])

    for i, heading in enumerate(headings):
        full_text, category, title = _extract_section_text(heading, section_map)

        if not full_text.strip() or len(full_text.strip()) < 30:
            continue

        full_text = _clean_code_tokenization(full_text)

        paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]
        current_chunk = ""

        for para in paragraphs:
            if estimate_tokens(current_chunk + " " + para) > CHUNK_MAX_TOKENS and current_chunk:
                chunk_counter += 1
                chunks.append({
                    "chunk_id": f"{source_type}_{chunk_counter:03d}",
                    "text": current_chunk.strip(),
                    "category": category,
                    "source_type": source_type,
                })
                current_chunk = para
            else:
                current_chunk = current_chunk + "\n\n" + para if current_chunk else para

        if current_chunk.strip() and estimate_tokens(current_chunk) >= CHUNK_MIN_TOKENS // 2:
            chunk_counter += 1
            chunks.append({
                "chunk_id": f"{source_type}_{chunk_counter:03d}",
                "text": current_chunk.strip(),
                "category": category,
                "source_type": source_type,
            })

            log.info(f" {source_type}: {len(chunks)} chunks extracted")
    save_cache(chunks, cache_key)
    return chunks


pep8_chunks = scrape_pep(
    "https://peps.python.org/pep-0008/",
    PEP8_SECTION_MAP,
    "pep8"
)

pep257_chunks = scrape_pep(
    "https://peps.python.org/pep-0257/",
    PEP257_SECTION_MAP,
    "pep257"
)

log.info(f"\nPEP chunks: {len(pep8_chunks)} (PEP8) + {len(pep257_chunks)} (PEP257) = {len(pep8_chunks) + len(pep257_chunks)} total")

pep_cats = Counter(c["category"] for c in pep8_chunks + pep257_chunks)
for cat, n in sorted(pep_cats.items()):
    log.info(f" {cat}: {n}")

In [ ]:
guideline_chunks = []
chunk_counter = 0

for repo_name in TRAIN_REPOS:
    log.info(f"\n Fetching contributing guide: {repo_name}")
    content = fetch_contributing_guide(repo_name)

    if not content:
        log.warning(f" ️ No contributing guide found for {repo_name}")
        continue

    sections = re.split(r"\n(?=#{1,4}\s|={3,}|-{3,}|\*{3,})", content)

    for section in sections:
        if len(section.strip()) < 50:
            continue

        section_lower = section.lower()
        if any(kw in section_lower for kw in ["indent", "tab", "space", "whitespace"]):
            category = "indentation"
        elif any(kw in section_lower for kw in ["naming", "name convention", "snake_case", "camelcase", "variable name"]):
            category = "naming_convention"
        elif any(kw in section_lower for kw in ["import", "unused", "dependency"]):
            category = "unused_import"
        elif any(kw in section_lower for kw in ["mutable", "default argument", "default parameter"]):
            category = "mutable_default"
        elif any(kw in section_lower for kw in ["docstring", "documentation", "comment", "format", "style guide", "coding style"]):
            category = "documentation_formatting"
        else:
            category = "general"

        paragraphs = [p.strip() for p in section.split("\n\n") if p.strip()]
        current_chunk = ""

        for para in paragraphs:
            if estimate_tokens(current_chunk + " " + para) > CHUNK_MAX_TOKENS and current_chunk:
                chunk_counter += 1
                guideline_chunks.append({
                    "chunk_id": f"guideline_{chunk_counter:03d}",
                    "text": current_chunk.strip(),
                    "category": category,
                    "source_type": "project_guideline",
                    "repo": repo_name,
                })
                current_chunk = para
            else:
                current_chunk = current_chunk + "\n\n" + para if current_chunk else para

        if current_chunk.strip() and estimate_tokens(current_chunk) >= CHUNK_MIN_TOKENS // 2:
            chunk_counter += 1
            guideline_chunks.append({
                "chunk_id": f"guideline_{chunk_counter:03d}",
                "text": current_chunk.strip(),
                "category": category,
                "source_type": "project_guideline",
                "repo": repo_name,
            })

log.info(f"\n Guideline chunks: {len(guideline_chunks)}")
cats = Counter(c["category"] for c in guideline_chunks)
for cat, n in sorted(cats.items()):
    log.info(f" {cat}: {n}")

In [ ]:
LINTER_RULES = [
    {
        "chunk_id": "linter_001",
        "text": "E111 (Flake8/pycodestyle): Indentation is not a multiple of the configured tabsize (default: 4 spaces). "
                "Python code should use 4 spaces per indentation level. Mixing tabs and spaces is not allowed in Python 3. "
                "Each nested block (if, for, while, def, class, with, try) should be indented by exactly 4 spaces.",
        "category": "indentation",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_002",
        "text": "E114 (Flake8): Indentation is not a multiple of four (comment). Comment lines should be indented "
                "to the same level as the code they describe. E117: Over-indented. E112: Expected an indented block. "
                "W191: Indentation contains tabs. All indentation should use spaces, not tabs.",
        "category": "indentation",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_003",
        "text": "E121-E131 (Flake8): Continuation line indentation issues. E121: continuation line under-indented for hanging indent. "
                "E122: missing indentation or outdented. E123: closing bracket does not match indentation. "
                "E125: continuation line with same indent as next logical line. E127: continuation line over-indented for visual indent. "
                "Continuation lines should align wrapped elements either using Python's implicit line joining inside parentheses, "
                "brackets and braces, or using a hanging indent.",
        "category": "indentation",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_004",
        "text": "C0103 (Pylint): Variable/function/class name doesn't conform to naming convention. "
                "Functions and variables should use snake_case (lowercase_with_underscores). "
                "Class names should use CapWords (PascalCase). Constants should use UPPER_CASE_WITH_UNDERSCORES. "
                "Module names should be short, all-lowercase, with underscores if needed.",
        "category": "naming_convention",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_005",
        "text": "N801-N818 (pep8-naming plugin): N801: class names should use CapWords convention. "
                "N802: function name should be lowercase. N803: argument name should be lowercase. "
                "N804: first argument of a classmethod should be named 'cls'. "
                "N805: first argument of a method should be named 'self'. "
                "N806: variable in function should be lowercase. "
                "N811-N818: various naming convention violations for constants, imports, and exceptions.",
        "category": "naming_convention",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_006",
        "text": "F401 (Flake8/Pyflakes): Module imported but unused. An import statement brings a module into "
                "the namespace but the module is never referenced in the file. Unused imports should be removed "
                "because they: (1) add unnecessary dependencies, (2) slow down module loading, "
                "(3) make code harder to read, (4) can cause confusion about what the module actually needs.",
        "category": "unused_import",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_007",
        "text": "W0611 (Pylint): Unused import. This is Pylint's equivalent of Flake8's F401. "
                "It detects modules or names imported but never used in the current file. "
                "Common causes: (1) leftover imports from refactoring, (2) importing for side effects without a noqa comment, "
                "(3) importing for type checking without TYPE_CHECKING guard. "
                "Fix: remove the import, or add '# noqa: F401' if imported for side effects.",
        "category": "unused_import",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_008",
        "text": "W0102 (Pylint): Dangerous default value (e.g., [] or {} or set()) as argument. "
                "Mutable default arguments are a common Python pitfall: the default object is shared across all calls "
                "to the function, so modifications persist between calls. "
                "Bad: def foo(items=[]) — Good: def foo(items=None): if items is None: items = [] "
                "This applies to lists, dicts, sets, and any other mutable object used as a default parameter.",
        "category": "mutable_default",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_009",
        "text": "B006 (flake8-bugbear): Do not use mutable data structures for argument defaults. "
                "Mutable default arguments are evaluated once at function definition time, not at each call. "
                "This means if you mutate the default argument, the next call will see the mutated value. "
                "Use None as default and create a new mutable inside the function body: "
                "def process(data=None): data = data or []",
        "category": "mutable_default",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_010",
        "text": "D100-D107 (pydocstyle): Missing docstrings. D100: missing module docstring. "
                "D101: missing docstring in public class. D102: missing docstring in public method. "
                "D103: missing docstring in public function. D104: missing docstring in public package. "
                "D105: missing docstring in magic method. D106: missing docstring in public nested class. "
                "D107: missing docstring in __init__. Every public module, class, function, and method should have a docstring.",
        "category": "documentation_formatting",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_011",
        "text": "D200-D215 (pydocstyle): Docstring formatting issues. D200: no blank lines allowed after function docstring. "
                "D205: 1 blank line required between summary line and description. D210: no whitespaces allowed surrounding docstring text. "
                "D300: use triple double quotes for docstrings. D400: first line should end with a period. "
                "D401: first line should be in imperative mood. D403: first word of the first line should be properly capitalized.",
        "category": "documentation_formatting",
        "source_type": "linter_rule",
    },
    {
        "chunk_id": "linter_012",
        "text": "C0114-C0116 (Pylint): Missing module/class/function docstring. "
                "C0301: Line too long (>{max-line-length} characters). "
                "C0303: Trailing whitespace. C0304: Final newline missing. "
                "W0105: String statement has no effect — often a multi-line string used as a comment instead of a proper docstring. "
                "Good documentation follows numpydoc, Google, or Sphinx style consistently.",
        "category": "documentation_formatting",
        "source_type": "linter_rule",
    },
]

log.info(f" Linter rules: {len(LINTER_RULES)} curated entries")
cats = Counter(r["category"] for r in LINTER_RULES)
for cat, n in sorted(cats.items()):
    log.info(f" {cat}: {n}")

In [ ]:
retrieval_corpus = []

pep8_relevant = [c for c in pep8_chunks if c["category"] != "general"]
retrieval_corpus.extend(pep8_relevant)
log.info(f"PEP8: kept {len(pep8_relevant)}/{len(pep8_chunks)} (removed {len(pep8_chunks)-len(pep8_relevant)} general chunks)")

pep257_relevant = [c for c in pep257_chunks if c["category"] != "general"]
retrieval_corpus.extend(pep257_relevant)
log.info(f"PEP257: kept {len(pep257_relevant)}/{len(pep257_chunks)} chunks")

guideline_relevant = [c for c in guideline_chunks if c["category"] != "general"]
retrieval_corpus.extend(guideline_relevant)
log.info(f"Guidelines: kept {len(guideline_relevant)}/{len(guideline_chunks)} (removed {len(guideline_chunks)-len(guideline_relevant)} general)")

retrieval_corpus.extend(LINTER_RULES)

mutable_default_knowledge = [
    {
        "chunk_id": "md_bp_001",
        "text": (
            "Mutable default arguments in Python are a common source of bugs. "
            "When a mutable object like a list, dictionary, or set is used as a "
            "default argument value, the same object is reused across all calls "
            "to the function. This means mutations from one call persist into "
            "subsequent calls.\n\n"
            "Bad example:\n"
            "```python\n"
            "def append_to(element, target=[]):\n"
            "    target.append(element)\n"
            "    return target\n"
            "```\n\n"
            "append_to(1)  # returns [1]\n"
            "append_to(2)  # returns [1, 2] — NOT [2]!\n\n"
            "Correct pattern:\n"
            "```python\n"
            "def append_to(element, target=None):\n"
            "    if target is None:\n"
            "        target = []\n"
            "    target.append(element)\n"
            "    return target\n"
            "```"
        ),
        "category": "mutable_default",
        "source_type": "best_practice",
    },
    {
        "chunk_id": "md_bp_002",
        "text": (
            "Common mutable default argument patterns to avoid:\n\n"
            "1. def func(items=[]): — use def func(items=None):\n"
            "2. def func(config={}): — use def func(config=None):\n"
            "3. def func(seen=set()): — use def func(seen=None):\n"
            "4. def func(cache={'key': []}): — use None pattern\n"
            "5. Class attributes: class Foo: items = [] in __init__ params\n\n"
            "The None sentinel pattern:\n"
            "```python\n"
            "def func(items=None):\n"
            "    if items is None:\n"
            "        items = []\n"
            "    # now safe to mutate items\n"
            "```\n\n"
            "This is documented in Python's official FAQ and flagged by "
            "Pylint W0102 and flake8-bugbear B006."
        ),
        "category": "mutable_default",
        "source_type": "best_practice",
    },
    {
        "chunk_id": "md_bp_003",
        "text": (
            "Why mutable default arguments are dangerous:\n\n"
            "Default argument values are evaluated once when the function "
            "definition is executed, not each time the function is called. "
            "If the default is a mutable object (list, dict, set, bytearray), "
            "it is shared between all calls that don't provide that argument.\n\n"
            "Python functions are objects, and default values are stored as "
            "attributes of the function object (in __defaults__ or __kwdefaults__).\n\n"
            "```python\n"
            "def f(x=[]):\n"
            "    x.append(1)\n"
            "    return x\n\n"
            "f()  # [1]\n"
            "f()  # [1, 1]\n"
            "f.__defaults__  # ([1, 1],) — the same list!\n"
            "```\n\n"
            "Immutable defaults (None, True, False, integers, strings, tuples) "
            "are safe because they cannot be modified in place."
        ),
        "category": "mutable_default",
        "source_type": "best_practice",
    },
]
retrieval_corpus.extend(mutable_default_knowledge)

review_chunk_counter = 0
for repo_name in TRAIN_REPOS:
    sampled = all_sampled.get(repo_name, [])
    high_quality = [
        c for c in sampled
        if c["violation_category"] in CATEGORIES
    ]

    for comment in high_quality:
        review_chunk_counter += 1
        text = f"Review comment on {comment['path']}:\n{comment['body']}"
        if comment.get("diff_hunk"):
            text += f"\n\nCode context:\n{comment['diff_hunk'][:500]}"

        retrieval_corpus.append({
            "chunk_id": f"review_{review_chunk_counter:04d}",
            "text": text,
            "category": comment["violation_category"],
            "source_type": "review_comment",
            "repo": repo_name,
        })

log.info(f"Review comment chunks added: {review_chunk_counter} from train repos")

_cleaned_count = 0
for chunk in retrieval_corpus:
    original = chunk["text"]
    cleaned = _clean_code_tokenization(original)
    if cleaned != original:
        chunk["text"] = cleaned
        _cleaned_count += 1
log.info(f"Text cleaning: fixed tokenization in {_cleaned_count}/{len(retrieval_corpus)} chunks")

seen_texts = set()
deduped_corpus = []
dupes_removed = 0
for chunk in retrieval_corpus:
    text_key = chunk["text"][:200].strip().lower()
    if text_key not in seen_texts:
        seen_texts.add(text_key)
        deduped_corpus.append(chunk)
    else:
        dupes_removed += 1

retrieval_corpus = deduped_corpus
log.info(f"Deduplication: removed {dupes_removed} duplicate chunks")

for i, chunk in enumerate(retrieval_corpus):
    chunk["chunk_id"] = f"chunk_{i+1:04d}"

general_count = sum(1 for c in retrieval_corpus if c["category"] == "general")
if general_count > 0:
    log.warning(f"️ {general_count} 'general' chunks still remain — removing...")
    retrieval_corpus = [c for c in retrieval_corpus if c["category"] != "general"]
    for i, chunk in enumerate(retrieval_corpus):
        chunk["chunk_id"] = f"chunk_{i+1:04d}"

with open(RETRIEVAL_CORPUS_PATH, "w") as f:
    json.dump(retrieval_corpus, f, indent=2, ensure_ascii=False)

log.info(f"\n{'='*60}")
log.info(f" retrieval_corpus.json saved → {RETRIEVAL_CORPUS_PATH}")
log.info(f" Total chunks: {len(retrieval_corpus)} (0 general pollution)")
log.info(f"{'='*60}")

corpus_df = pd.DataFrame(retrieval_corpus)
log.info("\nBy source_type:")
display(corpus_df["source_type"].value_counts().to_frame())
log.info("\nBy category:")
display(corpus_df["category"].value_counts().to_frame())

In [ ]:
evaluation_dataset = []
_eval_filtered_line0 = 0
_eval_deduped_gt = 0

for repo_name in EVAL_REPOS:
    sampled = all_sampled.get(repo_name, [])
    if not sampled:
        log.warning(f"️ No sampled data for {repo_name}")
        continue

    by_pr = defaultdict(list)
    for c in sampled:
        if c["violation_category"] in CATEGORIES:
            by_pr[c["pr_number"]].append(c)

    for pr_number, pr_comments in by_pr.items():
        by_file = defaultdict(list)
        for c in pr_comments:
            by_file[c["path"]].append(c)

        for file_path, file_comments in by_file.items():
            pr_files = all_pr_files.get((repo_name, pr_number), [])
            matching_file = next((f for f in pr_files if f["filename"] == file_path), None)

            diff_chunks = []
            if matching_file and matching_file.get("patch"):
                hunks = parse_diff_hunks(matching_file["patch"])
                for j, hunk in enumerate(hunks):
                    diff_chunks.append({
                        "chunk_id": f"c{j+1}",
                        "start_line": hunk["new_start"],
                        "end_line": hunk["new_end"],
                        "diff_lines": hunk["lines"],
                    })
            else:
                for j, c in enumerate(file_comments):
                    if c.get("diff_hunk"):
                        diff_chunks.append({
                            "chunk_id": f"c{j+1}",
                            "start_line": c.get("line", 0) or 0,
                            "end_line": c.get("line", 0) or 0,
                            "diff_lines": c["diff_hunk"].split("\n"),
                        })

            ground_truth = []
            seen_gt_keys = set()
            for c in file_comments:
                line_num = c.get("line") or c.get("original_line") or 0
                
                if line_num == 0:
                    _eval_filtered_line0 += 1
                    continue
                
                gt_key = (line_num, c["violation_category"])
                if gt_key in seen_gt_keys:
                    _eval_deduped_gt += 1
                    continue
                seen_gt_keys.add(gt_key)
                
                ground_truth.append({
                    "line_number": line_num,
                    "violation_category": c["violation_category"],
                    "review_comment": c["body"],
                })

            if not ground_truth:
                continue

            entry = {
                "pr_id": f"PR_{pr_number}",
                "repo": repo_name,
                "file_path": file_path,
                "diff_chunks": diff_chunks,
                "ground_truth_reviews": ground_truth,
            }
            evaluation_dataset.append(entry)

with open(EVAL_DATASET_PATH, "w") as f:
    json.dump(evaluation_dataset, f, indent=2, ensure_ascii=False)

log.info(f"\n{'='*60}")
log.info(f" evaluation_dataset.json saved → {EVAL_DATASET_PATH}")
log.info(f" Total entries: {len(evaluation_dataset)}")
log.info(f" Filtered line_number=0: {_eval_filtered_line0}")
log.info(f" Deduplicated GT entries: {_eval_deduped_gt}")
log.info(f"{'='*60}")

total_violations = sum(len(e["ground_truth_reviews"]) for e in evaluation_dataset)
log.info(f" Total violations: {total_violations}")

eval_cats = Counter()
eval_repos_counts = Counter()
for entry in evaluation_dataset:
    for gt in entry["ground_truth_reviews"]:
        eval_cats[gt["violation_category"]] += 1
        eval_repos_counts[entry["repo"]] += 1

log.info(f"\n By category:")
for cat, n in sorted(eval_cats.items()):
    log.info(f" {cat}: {n}")

log.info(f"\n By repo:")
for repo, n in sorted(eval_repos_counts.items()):
    log.info(f" {repo}: {n}")

In [ ]:
def fetch_full_file_content(repo_name: str, file_path: str, pr_number: int) -> Optional[str]:
    """
    Fetch the FULL source file content from the PR's head branch.
    Falls back to default branch if head ref is gone.
    """
    cache_key = f"full_file_{repo_name.replace('/', '_')}_{pr_number}_{file_path.replace('/', '_')}"
    cached = load_cache(cache_key)
    if cached is not None:
        return cached[0] if cached else None
    
    repo = rate_limiter.safe_call(gh.get_repo, repo_name)
    
    try:
        pr = rate_limiter.safe_call(repo.get_pull, pr_number)
        ref = pr.merge_commit_sha or pr.head.sha
        if ref:
            try:
                content_file = rate_limiter.safe_call(repo.get_contents, file_path, ref=ref)
                if content_file and hasattr(content_file, 'decoded_content'):
                    content = content_file.decoded_content.decode('utf-8', errors='replace')
                    save_cache([content], cache_key)
                    return content
            except GithubException:
                pass
    except Exception:
        pass
    
    try:
        content_file = rate_limiter.safe_call(repo.get_contents, file_path)
        if content_file and hasattr(content_file, 'decoded_content'):
            content = content_file.decoded_content.decode('utf-8', errors='replace')
            save_cache([content], cache_key)
            return content
    except GithubException:
        pass
    
    save_cache([], cache_key)
    return None


def _compute_modified_lines(diff_chunks: list[dict]) -> list[int]:
    """
    Extract line numbers of added/modified lines from diff chunks.
    Parses diff_lines to find '+' lines and computes their absolute line numbers.
    """
    modified = set()
    for chunk in diff_chunks:
        start_line = chunk.get("start_line", 0)
        if start_line <= 0:
            continue
        
        current_new_line = start_line
        for line in chunk.get("diff_lines", []):
            if line.startswith("@@"):
                import re as _re
                match = _re.search(r'\+(\d+)', line)
                if match:
                    current_new_line = int(match.group(1))
                continue
            elif line.startswith("+"):
                modified.add(current_new_line)
                current_new_line += 1
            elif line.startswith("-"):
                continue
            elif line == "\\ No newline at end of file":
                continue
            else:
                current_new_line += 1
    
    return sorted(modified)


static_analysis_input = []
seen_files = set()
fetched = 0
fallback_diff = 0

for entry in evaluation_dataset:
    file_key = f"{entry['repo']}:{entry['file_path']}:{entry['pr_id']}"
    if file_key in seen_files:
        continue
    seen_files.add(file_key)

    pr_number = int(entry['pr_id'].replace('PR_', ''))
    
    modified_lines = _compute_modified_lines(entry.get("diff_chunks", []))
    
    full_content = fetch_full_file_content(entry['repo'], entry['file_path'], pr_number)
    
    if full_content:
        fetched += 1
        static_analysis_input.append({
            "file_path": entry["file_path"],
            "source_code": full_content,
            "source_type": "full_file",
            "pr_id": entry["pr_id"],
            "repo": entry["repo"],
            "modified_lines": modified_lines,
        })
    else:
        fallback_diff += 1
        code_lines = []
        for chunk in entry["diff_chunks"]:
            for line in chunk.get("diff_lines", []):
                if line.startswith("+"):
                    code_lines.append(line[1:])
                elif line.startswith("-") or line.startswith("@@") or line == "\\ No newline at end of file":
                    continue
                else:
                    code_lines.append(line)

        if code_lines:
            static_analysis_input.append({
                "file_path": entry["file_path"],
                "source_code": "\n".join(code_lines),
                "source_type": "reconstructed_diff",
                "pr_id": entry["pr_id"],
                "repo": entry["repo"],
                "modified_lines": list(range(1, len(code_lines) + 1)),
            })

with open(STATIC_ANALYSIS_PATH, "w") as f:
    json.dump(static_analysis_input, f, indent=2, ensure_ascii=False)

log.info(f"\n{'='*60}")
log.info(f" static_analysis_input.json saved → {STATIC_ANALYSIS_PATH}")
log.info(f" Total files: {len(static_analysis_input)}")
log.info(f" Full source fetched: {fetched} | Diff fallback: {fallback_diff}")
total_loc = sum(len(s['source_code'].split(chr(10))) for s in static_analysis_input)
total_modified = sum(len(s.get('modified_lines', [])) for s in static_analysis_input)
log.info(f" Total lines of code: {total_loc}")
log.info(f" Total modified lines: {total_modified}")
log.info(f"{'='*60}")

In [ ]:
VERIFY_SYSTEM_PROMPT = """You are a STRICT Python code review classifier. For each review comment, classify it into exactly ONE category:

- indentation: about indent levels, tabs vs spaces, alignment, continuation lines, whitespace structure
- naming_convention: about variable/function/class/module naming (snake_case, CamelCase, PEP8 naming)
- unused_import: about imports that are never used, redundant imports, removing imports
- mutable_default: about mutable default arguments like def foo(x=[]) or def bar(d={})
- documentation_formatting: about docstrings, documentation, doc formatting, comments, doc style
- none: EVERYTHING ELSE — bugs, logic, performance, testing, refactoring, architecture, API design, type hints, etc.

Default to "none" when uncertain. False negatives are better than false positives.

Respond with ONLY a JSON array of category strings in the SAME ORDER as the input.
Example input: [{"id":0,"body":"fix indent to 4 spaces"},{"id":1,"body":"good refactor"}]
Example output: ["indentation","none"]
No markdown, no explanation — just the raw JSON array."""

VERIFY_BATCH_SIZE = 15  # Smaller batches for reliable parsing

def verify_ground_truth(eval_data: list[dict]) -> list[dict]:
    """
    Cross-validate eval dataset labels with conservative rules.
    Uses independent re-classification (model 2 doesn't see original labels).
    """
    if len(available_models) < 2:
        log.warning("️ Only 1 model available — skipping cross-validation (need 2+ models)")
        return eval_data
    
    verify_model = available_models[1]
    verify_client = llm_clients[verify_model]
    log.info(f" Independent re-classification with {verify_model}...")
    
    all_violations = []
    for entry_idx, entry in enumerate(eval_data):
        for gt_idx, gt in enumerate(entry["ground_truth_reviews"]):
            all_violations.append({
                "entry_idx": entry_idx,
                "gt_idx": gt_idx,
                "body": gt["review_comment"][:500],
                "original_category": gt["violation_category"],
                "file": entry["file_path"],
            })
    
            log.info(f" Re-classifying {len(all_violations)} violations independently...")
    
    cat_counts = Counter(v["original_category"] for v in all_violations)
    median_count = sorted(cat_counts.values())[len(cat_counts) // 2] if cat_counts else 5
    
    reclassified_labels = []
    n_batches = (len(all_violations) + VERIFY_BATCH_SIZE - 1) // VERIFY_BATCH_SIZE
    
    for b_idx in range(n_batches):
        start = b_idx * VERIFY_BATCH_SIZE
        end = min(start + VERIFY_BATCH_SIZE, len(all_violations))
        batch = all_violations[start:end]
        
        items = [{"id": i, "body": v["body"], "code_file": v["file"]} for i, v in enumerate(batch)]
        user_msg = json.dumps(items, ensure_ascii=False)
        
        time.sleep(60.0 / LLM_RPM_LIMIT + 0.5)
        
        success = False
        for attempt in range(3):
            try:
                response = verify_client.chat.completions.create(
                    model=verify_model,
                    messages=[
                        {"role": "system", "content": VERIFY_SYSTEM_PROMPT},
                        {"role": "user", "content": user_msg},
                    ],
                    max_tokens=len(batch) * 25,
                    temperature=0.0,
                )
                raw = response.choices[0].message.content.strip()
                if raw.startswith("```"):
                    raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
                labels = json.loads(raw)
                if isinstance(labels, list) and len(labels) == len(batch):
                    reclassified_labels.extend([
                        lbl.strip().lower() if isinstance(lbl, str) and lbl.strip().lower() in VALID_LABELS else "none"
                        for lbl in labels
                    ])
                    success = True
                    break
                else:
                    log.warning(f" Batch {b_idx+1} attempt {attempt+1}: length mismatch ({len(labels)} vs {len(batch)})")
                    time.sleep(2)
            except json.JSONDecodeError:
                log.warning(f" Batch {b_idx+1} attempt {attempt+1}: JSON parse error")
                time.sleep(2)
            except Exception as e:
                log.warning(f" Batch {b_idx+1} attempt {attempt+1}: {e}")
                time.sleep(4)
        
        if not success:
            log.warning(f" ️ Batch {b_idx+1} failed — keeping original labels (conservative)")
            reclassified_labels.extend([v["original_category"] for v in batch])
        
            log.info(f" Verified batch {b_idx+1}/{n_batches}")
    
    agreements = 0
    corrections = 0
    removals = 0
    kept_despite_disagreement = 0
    entries_to_remove_gt = []
    
    remaining_cats = Counter(v["original_category"] for v in all_violations)
    MIN_PER_CATEGORY = 2  # Never let a category drop below this
    
    for v_item, new_label in zip(all_violations, reclassified_labels):
        entry_idx = v_item["entry_idx"]
        gt_idx = v_item["gt_idx"]
        original = v_item["original_category"]
        
        if new_label == original:
            agreements += 1
        elif new_label in CATEGORIES and new_label != "none":
            kept_despite_disagreement += 1
        elif new_label == "none":
            if remaining_cats[original] > MIN_PER_CATEGORY and remaining_cats[original] > median_count * 0.5:
                entries_to_remove_gt.append((entry_idx, gt_idx))
                remaining_cats[original] -= 1
                removals += 1
            else:
                kept_despite_disagreement += 1
        else:
            kept_despite_disagreement += 1
    
    for entry_idx, gt_idx in sorted(entries_to_remove_gt, reverse=True):
        eval_data[entry_idx]["ground_truth_reviews"].pop(gt_idx)
    
    eval_data = [e for e in eval_data if len(e["ground_truth_reviews"]) > 0]
    
    total_remaining = sum(len(e["ground_truth_reviews"]) for e in eval_data)
    log.info(f"\n Conservative verification complete ({verify_model}):")
    log.info(f" Agreements (both models match): {agreements}")
    log.info(f" Kept despite disagreement: {kept_despite_disagreement}")
    log.info(f" Removals (confident none): {removals}")
    log.info(f" Remaining violations: {total_remaining} across {len(eval_data)} entries")
    
    return eval_data


evaluation_dataset = verify_ground_truth(evaluation_dataset)

with open(EVAL_DATASET_PATH, "w") as f:
    json.dump(evaluation_dataset, f, indent=2, ensure_ascii=False)

total_violations = sum(len(e["ground_truth_reviews"]) for e in evaluation_dataset)
log.info(f"\n Verified evaluation_dataset.json saved — {len(evaluation_dataset)} entries, {total_violations} violations")

eval_cats = Counter()
for entry in evaluation_dataset:
    for gt in entry["ground_truth_reviews"]:
        eval_cats[gt["violation_category"]] += 1
log.info(f" Verified category distribution:")
for cat, n in sorted(eval_cats.items()):
    log.info(f" {cat}: {n}")

In [ ]:
import os, json, time, random, copy, logging
from pathlib import Path
from collections import Counter
from openai import OpenAI

random.seed(42)

log = logging.getLogger("datascrape")

load_dotenv()
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
CATEGORIES = ["indentation", "naming_convention", "unused_import", "mutable_default", "documentation_formatting"]
LLM_RPM_LIMIT = 15

OUTPUT_DIR = Path(".")
EVAL_DATASET_PATH = OUTPUT_DIR / "evaluation_dataset.json"
STATIC_ANALYSIS_PATH = OUTPUT_DIR / "static_analysis_input.json"

MODEL_POOL = ["gpt-4o", "gpt-4.1-mini"]
llm_clients = {}
available_models = []

for model_name in MODEL_POOL:
    client = OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKEN,
    )
    try:
        _test = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=5,
        )
        llm_clients[model_name] = client
        available_models.append(model_name)
        log.info(f" {model_name} ready")
    except Exception as e:
        log.warning(f" ️ {model_name} unavailable: {e}")

_model_index = 0
def _get_next_model():
    global _model_index
    model = available_models[_model_index % len(available_models)]
    _model_index += 1
    return model, llm_clients[model]

log.info(f"Models available: {available_models}")

with open(EVAL_DATASET_PATH) as f:
    evaluation_dataset = json.load(f)
with open(STATIC_ANALYSIS_PATH) as f:
    static_analysis_input = json.load(f)

log.info(f"Loaded eval: {len(evaluation_dataset)} entries, static: {len(static_analysis_input)} files")

TARGET_PER_CAT = 30

organic_cat_counts = Counter()
for entry in evaluation_dataset:
    for gt in entry["ground_truth_reviews"]:
        organic_cat_counts[gt["violation_category"]] += 1

log.info(f"Organic eval violations: {dict(organic_cat_counts)} (total {sum(organic_cat_counts.values())})")
deficit = {cat: max(0, TARGET_PER_CAT - organic_cat_counts.get(cat, 0)) for cat in CATEGORIES}
log.info(f"Deficit per category: {deficit}")
total_needed = sum(deficit.values())
log.info(f"Total synthetic entries needed: {total_needed}")

if total_needed == 0:
    log.info(" No synthetic eval entries needed — targets already met!")
else:
    SYNTH_EVAL_PROMPT = """You are a Python code generator for a PEP 8 violation dataset.

Generate EXACTLY {batch_size} SHORT, REALISTIC Python code snippets (15-40 lines each).
Each snippet MUST contain ONE intentional violation from this category: **{category}**

Category definitions:
- indentation: Wrong indent level (3 spaces, 5 spaces, mixed tabs/spaces, misaligned continuation)
- naming_convention: camelCase functions, UPPERCASE variables, single-letter names in non-loop contexts, non-PEP8 class names
- unused_import: Import a module (os, sys, json, re, typing, etc.) that is NEVER used in the code body
- mutable_default: Function with def foo(items=[], config={{}}, seen=set()) — mutable default arg
- documentation_formatting: Missing docstring on public function/class, malformed docstring (no summary line, wrong indentation, missing Args/Returns)

Requirements:
- Each snippet must be a complete, runnable Python module or function definition
- Use realistic variable/function names (not toy examples like foo/bar)
- The violation must be SUBTLE and realistic — something a real developer would write
- Do NOT include any comments explaining the violation
- Vary the code themes: web handlers, data processing, file I/O, API clients, DB queries, utilities, CLI tools, ML pipelines, etc.

Output format: Return a JSON array of objects, each with:
  "code": the Python code as a string (use \\n for newlines),
  "violation_line": the 1-based line number where the violation occurs,
  "review_comment": a realistic reviewer comment pointing out the violation (1-2 sentences),
  "file_name": a realistic Python filename (e.g., "utils/cache_manager.py")

Return ONLY the JSON array, no markdown fences, no explanation."""

    SYNTH_BATCH_SIZE = 5  # 5 snippets per API call for reliability

    synthetic_eval_entries = []
    synthetic_static_entries = []
    synth_counter = 0

    for cat in CATEGORIES:
        needed = deficit[cat]
        if needed <= 0:
            continue

        log.info(f"\n Generating {needed} synthetic entries for: {cat}")
        generated = 0
        attempts = 0
        max_attempts = (needed // SYNTH_BATCH_SIZE + 2) * 3  # generous retry budget

        while generated < needed and attempts < max_attempts:
            attempts += 1
            batch_size = min(SYNTH_BATCH_SIZE, needed - generated)

            model_name, client = _get_next_model()

            prompt = SYNTH_EVAL_PROMPT.format(batch_size=batch_size, category=cat)

            time.sleep(60.0 / LLM_RPM_LIMIT + 0.5)

            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": "You are a helpful code generation assistant."},
                        {"role": "user", "content": prompt},
                    ],
                    max_tokens=3000,
                    temperature=0.8,
                )
                raw = response.choices[0].message.content.strip()

                if raw.startswith("```"):
                    raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

                snippets = json.loads(raw)
                if not isinstance(snippets, list):
                    log.warning(f" Attempt {attempts}: not a list")
                    continue

                for snip in snippets:
                    if generated >= needed:
                        break
                    code = snip.get("code", "")
                    v_line = snip.get("violation_line", 1)
                    comment = snip.get("review_comment", "")
                    fname = snip.get("file_name", f"synthetic/module_{synth_counter}.py")

                    if not code or not comment or len(code) < 30:
                        continue

                    synth_counter += 1
                    pr_id = f"SYNTH_{synth_counter:04d}"

                    code_lines = code.split("\n")
                    diff_lines = [f"+{line}" for line in code_lines]

                    eval_entry = {
                        "pr_id": pr_id,
                        "repo": "synthetic/pep8-violations",
                        "file_path": fname,
                        "origin": "synthetic",
                        "diff_chunks": [{
                            "chunk_id": "c1",
                            "start_line": 1,
                            "end_line": len(code_lines),
                            "diff_lines": diff_lines,
                        }],
                        "ground_truth_reviews": [{
                            "line_number": int(v_line) if isinstance(v_line, (int, float)) else 1,
                            "violation_category": cat,
                            "review_comment": comment,
                        }],
                    }
                    synthetic_eval_entries.append(eval_entry)

                    static_entry = {
                        "file_path": fname,
                        "source_code": code,
                        "source_type": "synthetic",
                        "pr_id": pr_id,
                        "repo": "synthetic/pep8-violations",
                        "origin": "synthetic",
                        "modified_lines": list(range(1, len(code_lines) + 1)),
                    }
                    synthetic_static_entries.append(static_entry)

                    generated += 1

                    log.info(f" {cat}: {generated}/{needed} generated (attempt {attempts}, model={model_name})")

            except json.JSONDecodeError as e:
                log.warning(f" Attempt {attempts}: JSON parse error — {e}")
                time.sleep(2)
            except Exception as e:
                log.warning(f" Attempt {attempts}: {e}")
                time.sleep(4)

        if generated < needed:
            log.warning(f" ️ {cat}: only generated {generated}/{needed} after {attempts} attempts")

    evaluation_dataset.extend(synthetic_eval_entries)
    static_analysis_input.extend(synthetic_static_entries)

    with open(EVAL_DATASET_PATH, "w") as f:
        json.dump(evaluation_dataset, f, indent=2, ensure_ascii=False)

    with open(STATIC_ANALYSIS_PATH, "w") as f:
        json.dump(static_analysis_input, f, indent=2, ensure_ascii=False)

    total_eval_violations = sum(len(e["ground_truth_reviews"]) for e in evaluation_dataset)
    synth_violations = sum(len(e["ground_truth_reviews"]) for e in synthetic_eval_entries)

    log.info(f"\n{'='*60}")
    log.info(f" Synthetic augmentation complete!")
    log.info(f" Synthetic entries added: {len(synthetic_eval_entries)} eval, {len(synthetic_static_entries)} static")
    log.info(f" Synthetic violations: {synth_violations}")
    log.info(f" Total eval violations now: {total_eval_violations}")
    log.info(f" Total static files now: {len(static_analysis_input)}")
    log.info(f"{'='*60}")

    final_cats = Counter()
    for entry in evaluation_dataset:
        for gt in entry["ground_truth_reviews"]:
            final_cats[gt["violation_category"]] += 1
            log.info(f"\n Final category distribution (organic + synthetic):")
    for cat in CATEGORIES:
        org = organic_cat_counts.get(cat, 0)
        total = final_cats.get(cat, 0)
        syn = total - org
        log.info(f" {cat}: {total} (organic={org}, synthetic={syn})")

In [ ]:
import json, time, re as _re, logging
from pathlib import Path
from collections import Counter
from openai import OpenAI

log = logging.getLogger("datascrape")

load_dotenv()
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
CATEGORIES = ["indentation", "naming_convention", "unused_import", "mutable_default", "documentation_formatting"]
LLM_RPM_LIMIT = 15
CHUNK_MAX_TOKENS = 400

OUTPUT_DIR = Path(".")
RETRIEVAL_CORPUS_PATH = OUTPUT_DIR / "retrieval_corpus.json"

def estimate_tokens(text: str) -> int:
    return max(1, int(len(text.split()) * 1.3))

def _clean_code_tokenization(text: str) -> str:
    text = _re.sub(r'\s+\(', '(', text)
    text = _re.sub(r'\s+\)', ')', text)
    text = _re.sub(r'\s+\]', ']', text)
    text = _re.sub(r'\(\s+', '(', text)
    text = _re.sub(r'\[\s+', '[', text)
    text = _re.sub(r'\(\s*\)', '()', text)
    text = _re.sub(r'\[\s*\]', '[]', text)
    text = _re.sub(r'\{\s*\}', '{}', text)
    text = _re.sub(r'\s*,\s+', ', ', text)
    text = _re.sub(r'\s*:\s+', ': ', text)
    text = _re.sub(r'\s*\.\s*', '.', text)
    text = _re.sub(r'@\s+', '@', text)
    text = _re.sub(r'\s*=\s*', ' = ', text)
    text = _re.sub(r' = =', ' ==', text)
    text = _re.sub(r'! =', '!=', text)
    text = _re.sub(r'< =', '<=', text)
    text = _re.sub(r'> =', '>=', text)
    text = _re.sub(r'\+ =', '+=', text)
    text = _re.sub(r'- =', '-=', text)
    text = _re.sub(r'\* =', '*=', text)
    text = _re.sub(r'[^\S\n]+', ' ', text)
    return text

if 'available_models' not in dir() or not available_models:
    MODEL_POOL = ["gpt-4o", "gpt-4.1-mini"]
    llm_clients = {}
    available_models = []
    for model_name in MODEL_POOL:
        client = OpenAI(base_url="https://models.inference.ai.azure.com", api_key=GITHUB_TOKEN)
        try:
            client.chat.completions.create(model=model_name, messages=[{"role": "user", "content": "OK"}], max_tokens=5)
            llm_clients[model_name] = client
            available_models.append(model_name)
        except:
            pass

_model_index = 0
def _get_next_model():
    global _model_index
    model = available_models[_model_index % len(available_models)]
    _model_index += 1
    return model, llm_clients[model]

with open(RETRIEVAL_CORPUS_PATH) as f:
    retrieval_corpus = json.load(f)

log.info(f"Current corpus: {len(retrieval_corpus)} chunks")

rechunked_corpus = []
split_count = 0
for chunk in retrieval_corpus:
    tok_count = estimate_tokens(chunk["text"])
    if tok_count > CHUNK_MAX_TOKENS:
        paragraphs = [p.strip() for p in chunk["text"].split("\n\n") if p.strip()]
        current = ""
        for para in paragraphs:
            if estimate_tokens(current + "\n\n" + para) > CHUNK_MAX_TOKENS and current:
                rechunked_corpus.append({**chunk, "text": current.strip()})
                current = para
            else:
                current = (current + "\n\n" + para).strip() if current else para
        if current.strip():
            rechunked_corpus.append({**chunk, "text": current.strip()})
        split_count += 1
    else:
        rechunked_corpus.append(chunk)

log.info(f"Re-chunking: split {split_count} oversized chunks → {len(rechunked_corpus)} total")

CORPUS_TARGET_PER_CAT = 30
corpus_cats = Counter(c["category"] for c in rechunked_corpus)
log.info(f"Corpus category counts: {dict(corpus_cats)}")

corpus_deficit = {cat: max(0, CORPUS_TARGET_PER_CAT - corpus_cats.get(cat, 0)) for cat in CATEGORIES}
log.info(f"Corpus deficit per category: {corpus_deficit}")
total_corpus_needed = sum(corpus_deficit.values())

if total_corpus_needed == 0:
    log.info(" No synthetic corpus chunks needed — all categories ≥30!")
    retrieval_corpus = rechunked_corpus
else:
    CORPUS_GEN_PROMPT = """You are a Python best-practices knowledge base author.

Generate EXACTLY {n} self-contained knowledge chunks about **{category}** in Python.

Category definitions:
- indentation: PEP 8 indentation rules (4 spaces, continuation lines, hanging indents, alignment, tabs vs spaces)
- naming_convention: PEP 8 naming (snake_case, CamelCase, UPPER_CASE constants, _private, __dunder__, names to avoid)
- unused_import: Best practices for Python imports (organizing, removing unused, isort, __all__, lazy imports)
- mutable_default: Dangers of mutable default arguments, the None sentinel pattern, __defaults__, Pylint W0102, flake8-bugbear B006
- documentation_formatting: Docstring conventions (PEP 257), Google/NumPy/Sphinx styles, module/class/function docstrings, Args/Returns formatting

Requirements for EACH chunk:
- 150-350 words (200-400 tokens)
- Include at least one Python code example (```python ... ```)
- Must be factually correct and cite PEP numbers or tool names where relevant
- Written as a reference article paragraph, not a conversation
- Each chunk should cover a DIFFERENT sub-topic within the category
- NO overlap between chunks

Output: JSON array of objects with:
  "text": the knowledge chunk text (use \\n for newlines)

Return ONLY the JSON array."""

    synthetic_corpus_chunks = []
    for cat in CATEGORIES:
        needed = corpus_deficit[cat]
        if needed <= 0:
            continue

        log.info(f"\n Generating {needed} corpus chunks for: {cat}")
        generated = 0
        attempts = 0
        max_attempts = (needed // 5 + 2) * 3

        while generated < needed and attempts < max_attempts:
            attempts += 1
            batch_n = min(5, needed - generated)
            model_name, client = _get_next_model()

            prompt = CORPUS_GEN_PROMPT.format(n=batch_n, category=cat)
            time.sleep(60.0 / LLM_RPM_LIMIT + 0.5)

            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": "You are a technical documentation writer."},
                        {"role": "user", "content": prompt},
                    ],
                    max_tokens=3000,
                    temperature=0.7,
                )
                raw = response.choices[0].message.content.strip()
                if raw.startswith("```"):
                    raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

                chunks_data = json.loads(raw)
                if not isinstance(chunks_data, list):
                    continue

                for item in chunks_data:
                    if generated >= needed:
                        break
                    text = item.get("text", "")
                    if not text or len(text) < 50:
                        continue

                    text = _clean_code_tokenization(text)
                    tok = estimate_tokens(text)
                    if tok > CHUNK_MAX_TOKENS:
                        words = text.split()
                        text = " ".join(words[:int(len(words) * CHUNK_MAX_TOKENS / tok)])

                    synthetic_corpus_chunks.append({
                        "chunk_id": f"synth_corpus_{len(synthetic_corpus_chunks)+1:04d}",
                        "text": text,
                        "category": cat,
                        "source_type": "synthetic_knowledge",
                        "origin": "synthetic",
                    })
                    generated += 1

                    log.info(f" {cat}: {generated}/{needed} chunks (attempt {attempts}, model={model_name})")

            except json.JSONDecodeError:
                log.warning(f" Attempt {attempts}: JSON parse error")
                time.sleep(2)
            except Exception as e:
                log.warning(f" Attempt {attempts}: {e}")
                time.sleep(4)

        if generated < needed:
            log.warning(f" ️ {cat}: only generated {generated}/{needed}")

    retrieval_corpus = rechunked_corpus + synthetic_corpus_chunks

for i, chunk in enumerate(retrieval_corpus):
    chunk["chunk_id"] = f"chunk_{i+1:04d}"

seen_texts = set()
deduped = []
for chunk in retrieval_corpus:
    key = chunk["text"][:200].strip().lower()
    if key not in seen_texts:
        seen_texts.add(key)
        deduped.append(chunk)
retrieval_corpus = deduped

for i, chunk in enumerate(retrieval_corpus):
    chunk["chunk_id"] = f"chunk_{i+1:04d}"

retrieval_corpus = [c for c in retrieval_corpus if c.get("category") != "general"]
for i, chunk in enumerate(retrieval_corpus):
    chunk["chunk_id"] = f"chunk_{i+1:04d}"

with open(RETRIEVAL_CORPUS_PATH, "w") as f:
    json.dump(retrieval_corpus, f, indent=2, ensure_ascii=False)

log.info(f"\n{'='*60}")
log.info(f" Corpus augmentation complete!")
log.info(f" Synthetic chunks added: {len(synthetic_corpus_chunks) if total_corpus_needed > 0 else 0}")
log.info(f" Total corpus chunks: {len(retrieval_corpus)}")
log.info(f"{'='*60}")

final_corpus_cats = Counter(c["category"] for c in retrieval_corpus)
final_corpus_sources = Counter(c["source_type"] for c in retrieval_corpus)
log.info(f"\n By category:")
for cat in CATEGORIES:
    log.info(f" {cat}: {final_corpus_cats.get(cat, 0)}")
log.info(f"\n By source_type:")
for src, n in final_corpus_sources.most_common():
    log.info(f" {src}: {n}")

oversized = sum(1 for c in retrieval_corpus if estimate_tokens(c["text"]) > CHUNK_MAX_TOKENS)
log.info(f"\n Chunks exceeding {CHUNK_MAX_TOKENS} tokens: {oversized}")
if oversized > 0:
    log.warning(f" ️ {oversized} chunks still oversized — consider further splitting")

In [ ]:
import json, logging
from pathlib import Path

log = logging.getLogger("datascrape")

OUTPUT_DIR = Path(".")
EVAL_DATASET_PATH = OUTPUT_DIR / "evaluation_dataset.json"
STATIC_ANALYSIS_PATH = OUTPUT_DIR / "static_analysis_input.json"
RETRIEVAL_CORPUS_PATH = OUTPUT_DIR / "retrieval_corpus.json"

with open(EVAL_DATASET_PATH) as f:
    evaluation_dataset = json.load(f)
with open(STATIC_ANALYSIS_PATH) as f:
    static_analysis_input = json.load(f)

log.info(f"Loaded eval: {len(evaluation_dataset)} entries, static: {len(static_analysis_input)} files")

stripped_count = 0
total_lines_removed = 0
for entry in evaluation_dataset:
    for chunk in entry.get("diff_chunks", []):
        original_lines = chunk.get("diff_lines", [])
        cleaned_lines = [line for line in original_lines if not line.startswith("@@")]
        removed = len(original_lines) - len(cleaned_lines)
        if removed > 0:
            chunk["diff_lines"] = cleaned_lines
            total_lines_removed += removed
            stripped_count += 1

log.info(f"Diff cleanup: stripped @@ headers from {stripped_count} chunks ({total_lines_removed} lines removed)")

organic_eval = 0
synthetic_eval = 0
for entry in evaluation_dataset:
    if "origin" not in entry:
        entry["origin"] = "organic"
        organic_eval += 1
    else:
        synthetic_eval += 1

organic_static = 0
synthetic_static = 0
for entry in static_analysis_input:
    if "origin" not in entry:
        entry["origin"] = "organic"
        organic_static += 1
    else:
        synthetic_static += 1

log.info(f"Origin tagging — Eval: {organic_eval} organic, {synthetic_eval} synthetic")
log.info(f"Origin tagging — Static: {organic_static} organic, {synthetic_static} synthetic")

eval_pr_ids = set(e["pr_id"] for e in evaluation_dataset)
static_pr_ids = set(s["pr_id"] for s in static_analysis_input)
missing_in_static = eval_pr_ids - static_pr_ids
missing_in_eval = static_pr_ids - eval_pr_ids

if missing_in_static:
    log.warning(f"️ {len(missing_in_static)} eval entries have no matching static entry")
if missing_in_eval:
    log.info(f"️ {len(missing_in_eval)} static entries have no matching eval entry (OK for multi-file PRs)")

with open(EVAL_DATASET_PATH, "w") as f:
    json.dump(evaluation_dataset, f, indent=2, ensure_ascii=False)

with open(STATIC_ANALYSIS_PATH, "w") as f:
    json.dump(static_analysis_input, f, indent=2, ensure_ascii=False)

with open(RETRIEVAL_CORPUS_PATH) as f:
    retrieval_corpus = json.load(f)

for chunk in retrieval_corpus:
    if "origin" not in chunk:
        chunk["origin"] = "organic"

with open(RETRIEVAL_CORPUS_PATH, "w") as f:
    json.dump(retrieval_corpus, f, indent=2, ensure_ascii=False)

total_eval = len(evaluation_dataset)
total_violations = sum(len(e["ground_truth_reviews"]) for e in evaluation_dataset)
total_static = len(static_analysis_input)
total_corpus = len(retrieval_corpus)

log.info(f"\n{'='*60}")
log.info(f" All datasets cleaned and polished!")
log.info(f" evaluation_dataset.json: {total_eval} entries, {total_violations} violations")
log.info(f" static_analysis_input.json: {total_static} files")
log.info(f" retrieval_corpus.json: {total_corpus} chunks")
log.info(f"{'='*60}")

In [ ]:
import json, re as _re
from collections import Counter, defaultdict
from pathlib import Path

TRAIN_REPOS      = ["django/django", "pandas-dev/pandas", "scikit-learn/scikit-learn"]
EVAL_REPOS       = ["pallets/flask", "fastapi/fastapi"]
ALL_REPOS        = TRAIN_REPOS + EVAL_REPOS
CATEGORIES       = ["indentation", "naming_convention", "unused_import",
                    "mutable_default", "documentation_formatting"]
available_models = []   # no LLM; cross-validation check will score conservatively
CHUNK_MAX_TOKENS = 400

def estimate_tokens(text: str) -> int:
    return max(1, int(len(text.split()) * 1.3))


In [ ]:
import json, re as _re
from pathlib import Path

OUTPUT_DIR            = Path(".")
RETRIEVAL_CORPUS_PATH = OUTPUT_DIR / "retrieval_corpus.json"
STATIC_ANALYSIS_PATH  = OUTPUT_DIR / "static_analysis_input.json"
CHUNK_MAX_TOKENS      = 400

def estimate_tokens(text: str) -> int:
    return max(1, int(len(text.split()) * 1.3))

with open(RETRIEVAL_CORPUS_PATH) as f:
    corpus = json.load(f)

truncated = 0
too_small = 0
for chunk in corpus:
    tok = estimate_tokens(chunk["text"])
    if tok > CHUNK_MAX_TOKENS:
        words = chunk["text"].split()
        target_words = int(len(words) * CHUNK_MAX_TOKENS / tok) - 5
        chunk["text"] = " ".join(words[:max(target_words, 1)])
        truncated += 1
    if estimate_tokens(chunk["text"]) < 20:
        too_small += 1

print(f"Corpus: truncated {truncated} oversized chunks, {too_small} very small chunks remain.")

with open(RETRIEVAL_CORPUS_PATH, "w") as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

with open(STATIC_ANALYSIS_PATH) as f:
    static_data = json.load(f)

patched = 0
for entry in static_data:
    if not entry.get("modified_lines"):
        code = entry.get("source_code", "")
        n_lines = max(1, len(code.splitlines()))
        entry["modified_lines"] = list(range(1, n_lines + 1))
        patched += 1

print(f"Static analysis: patched {patched} entries with missing modified_lines.")

with open(STATIC_ANALYSIS_PATH, "w") as f:
    json.dump(static_data, f, indent=2, ensure_ascii=False)

print(" Pre-validation polish complete.")


In [ ]:
print("=" * 70)
print(" VALIDATION REPORT")
print("=" * 70)

errors = []

print("\n 1. JSON Validity Check")
for name, path in [
    ("retrieval_corpus.json", RETRIEVAL_CORPUS_PATH),
    ("evaluation_dataset.json", EVAL_DATASET_PATH),
    ("static_analysis_input.json", STATIC_ANALYSIS_PATH),
]:
    try:
        with open(path) as f:
            data = json.load(f)
        size_kb = path.stat().st_size / 1024
        print(f" {name}: valid JSON ({len(data)} items, {size_kb:.1f} KB)")
    except Exception as e:
        errors.append(f"{name}: {e}")
        print(f" {name}: INVALID — {e}")

print("\n 2. Data Leakage Check")
with open(RETRIEVAL_CORPUS_PATH) as f:
    corpus = json.load(f)

corpus_repos = set()
for chunk in corpus:
    if "repo" in chunk:
        corpus_repos.add(chunk["repo"])

eval_repos_in_corpus = corpus_repos.intersection(set(EVAL_REPOS))
if eval_repos_in_corpus:
    errors.append(f"DATA LEAKAGE: eval repos found in retrieval corpus: {eval_repos_in_corpus}")
    print(f" LEAKAGE DETECTED: {eval_repos_in_corpus}")
else:
    print(f" No eval repo data in retrieval corpus")
    print(f" Corpus repos: {corpus_repos - {'N/A', None}}")

print("\n 3. General Pollution Check")
general_count = sum(1 for c in corpus if c.get("category") == "general")
if general_count == 0:
    print(f" No 'general' category chunks in corpus — zero pollution!")
else:
    errors.append(f"General pollution: {general_count} chunks with category='general'")
    print(f" {general_count} 'general' chunks found")

print("\n 4. Category Stratification")
with open(EVAL_DATASET_PATH) as f:
    eval_data = json.load(f)

eval_cat_counts = Counter()
eval_repo_cat = defaultdict(Counter)
eval_origin_counts = Counter(e.get("origin", "organic") for e in eval_data)
for entry in eval_data:
    for gt in entry["ground_truth_reviews"]:
        eval_cat_counts[gt["violation_category"]] += 1
        eval_repo_cat[entry.get("repo", "unknown")][gt["violation_category"]] += 1

total_eval_violations = sum(eval_cat_counts.values())
print(f"\n Evaluation Dataset — Total violations: {total_eval_violations}")
print(f" Entries: {len(eval_data)} (organic={eval_origin_counts.get('organic',0)}, synthetic={eval_origin_counts.get('synthetic',0)})")
print(f"\n {'Category':<28} {'Count':>6} {'Target':>6} {'Status'}")
print(f" {'─'*60}")
for cat in CATEGORIES:
    count = eval_cat_counts.get(cat, 0)
    target = 20  # Updated target: ≥20 per category
    status = "✅" if count >= target else "⚠️ below target"
    print(f" {cat:<28} {count:>6} {target:>6} {status}")

print(f"\n Per-Source Breakdown:")
for repo in sorted(set(e.get("repo", "unknown") for e in eval_data)):
    cats = eval_repo_cat.get(repo, {})
    total = sum(cats.values())
    print(f" {repo}: {total} total")
    for cat in CATEGORIES:
        n = cats.get(cat, 0)
        if n > 0:
            print(f" {cat}: {n}")

print(f"\n 5. Retrieval Corpus Stats")
corpus_cats = Counter(c["category"] for c in corpus)
corpus_sources = Counter(c["source_type"] for c in corpus)
corpus_origin = Counter(c.get("origin", "organic") for c in corpus)

print(f" Total chunks: {len(corpus)} (organic={corpus_origin.get('organic',0)}, synthetic={corpus_origin.get('synthetic',0)})")
print(f"\n By source_type:")
for src, n in corpus_sources.most_common():
    print(f" {src}: {n}")
print(f"\n By category:")
for cat, n in corpus_cats.most_common():
    print(f" {cat}: {n}")

print(f"\n 6. Static Analysis Input Stats")
with open(STATIC_ANALYSIS_PATH) as f:
    static_data = json.load(f)
total_lines = sum(len(s["source_code"].split("\n")) for s in static_data)
full_files = sum(1 for s in static_data if s.get("source_type") == "full_file")
diff_files = sum(1 for s in static_data if s.get("source_type") == "reconstructed_diff")
synth_files = sum(1 for s in static_data if s.get("source_type") == "synthetic")
has_modified_lines = sum(1 for s in static_data if s.get("modified_lines"))
total_modified = sum(len(s.get("modified_lines", [])) for s in static_data)
print(f" Total files: {len(static_data)} (full={full_files}, diff={diff_files}, synthetic={synth_files})")
print(f" Total lines of code: {total_lines}")
print(f" Files with modified_lines: {has_modified_lines}/{len(static_data)}")
print(f" Total modified lines tracked: {total_modified}")

print(f"\n 7. Schema Validation")

required_corpus_keys = {"chunk_id", "text", "category", "source_type"}
for i, chunk in enumerate(corpus[:5]):
    missing = required_corpus_keys - set(chunk.keys())
    if missing:
        errors.append(f"Corpus chunk {i} missing keys: {missing}")
print(f" retrieval_corpus.json schema OK" if not any("Corpus" in e for e in errors) else f" Schema errors in corpus")

required_eval_keys = {"pr_id", "repo", "file_path", "diff_chunks", "ground_truth_reviews"}
for i, entry in enumerate(eval_data[:5]):
    missing = required_eval_keys - set(entry.keys())
    if missing:
        errors.append(f"Eval entry {i} missing keys: {missing}")
    for gt in entry.get("ground_truth_reviews", []):
        gt_missing = {"line_number", "violation_category", "review_comment"} - set(gt.keys())
        if gt_missing:
            errors.append(f"Eval GT entry missing keys: {gt_missing}")
print(f" evaluation_dataset.json schema OK" if not any("Eval" in e for e in errors) else f" Schema errors in eval dataset")

required_static_keys = {"file_path", "source_code", "pr_id", "repo", "modified_lines"}
for i, entry in enumerate(static_data[:5]):
    missing = required_static_keys - set(entry.keys())
    if missing:
        errors.append(f"Static entry {i} missing keys: {missing}")
print(f" static_analysis_input.json schema OK" if not any("Static" in e for e in errors) else f" Schema errors in static analysis")

print(f"\n 8. Ground Truth Quality Checks")

line0_count = sum(
    1 for entry in eval_data
    for gt in entry["ground_truth_reviews"]
    if gt.get("line_number", 0) == 0
)
if line0_count == 0:
    print(f" No line_number=0 entries (all filtered)")
else:
    errors.append(f"Found {line0_count} entries with line_number=0")
    print(f" {line0_count} entries still have line_number=0")

dup_gt_count = 0
for entry in eval_data:
    gt_keys = [(gt["line_number"], gt["violation_category"]) for gt in entry["ground_truth_reviews"]]
    if len(gt_keys) != len(set(gt_keys)):
        dup_gt_count += len(gt_keys) - len(set(gt_keys))
if dup_gt_count == 0:
    print(f" No duplicate ground truth entries")
else:
    errors.append(f"Found {dup_gt_count} duplicate GT entries")
    print(f" {dup_gt_count} duplicate GT entries remain")

bad_token_count = sum(
    1 for c in corpus
    if "( " in c["text"] and " )" in c["text"] and "```" not in c["text"][:50]
)
if bad_token_count == 0:
    print(f" No bad tokenization artifacts in corpus")
else:
    print(f" ️ {bad_token_count} chunks may have tokenization issues")

print(f"\n 9. Diff Marker Check")
at_marker_count = 0
for entry in eval_data:
    for chunk in entry.get("diff_chunks", []):
        for line in chunk.get("diff_lines", []):
            if line.startswith("@@"):
                at_marker_count += 1
if at_marker_count == 0:
    print(f" No @@ hunk headers in diff_lines (all stripped)")
else:
    errors.append(f"Found {at_marker_count} @@ markers remaining in diff_lines")
    print(f" {at_marker_count} @@ hunk headers still in diff_lines")

print(f"\n 10. Corpus Chunk Size Check")
oversized_chunks = sum(1 for c in corpus if estimate_tokens(c["text"]) > CHUNK_MAX_TOKENS)
undersized_chunks = sum(1 for c in corpus if estimate_tokens(c["text"]) < 20)
if oversized_chunks == 0:
    print(f" All chunks ≤{CHUNK_MAX_TOKENS} tokens")
else:
    print(f" ️ {oversized_chunks} chunks exceed {CHUNK_MAX_TOKENS} tokens")
if undersized_chunks == 0:
    print(f" No trivially small chunks (<20 tokens)")
else:
    print(f" ️ {undersized_chunks} chunks are very small (<20 tokens)")

print(f"\n 11. Synthetic Data Ratio")
synth_eval_pct = eval_origin_counts.get("synthetic", 0) / max(len(eval_data), 1) * 100
synth_corpus_pct = corpus_origin.get("synthetic", 0) / max(len(corpus), 1) * 100
print(f" Eval: {synth_eval_pct:.0f}% synthetic ({eval_origin_counts.get('synthetic',0)}/{len(eval_data)})")
print(f" Corpus: {synth_corpus_pct:.0f}% synthetic ({corpus_origin.get('synthetic',0)}/{len(corpus)})")
if synth_eval_pct <= 80:
    print(f" Synthetic ratio within acceptable range (≤80%)")
else:
    print(f" ️ High synthetic ratio — consider scraping more organic data")

print(f"\n 12. Data Quality Score")
quality_score = 0
max_score = 15

if general_count == 0:
    quality_score += 1
    print(f" [1/1] Corpus: zero general pollution")
else:
    print(f" [0/1] Corpus: {general_count} general chunks")

if len(corpus) >= 150:
    quality_score += 1
    print(f" [1/1] Corpus: {len(corpus)} chunks (target: ≥150)")
else:
    print(f" [0/1] Corpus: only {len(corpus)} chunks (target: ≥150)")

min_corpus_cat = min(corpus_cats.get(c, 0) for c in CATEGORIES)
if min_corpus_cat >= 25:
    quality_score += 1
    print(f" [1/1] Corpus: all categories ≥25 (min={min_corpus_cat})")
else:
    print(f" [0/1] Corpus: min category has {min_corpus_cat} (target: ≥25)")

if total_eval_violations >= 100:
    quality_score += 1
    print(f" [1/1] Eval: {total_eval_violations} violations (target: ≥100)")
else:
    print(f" [0/1] Eval: only {total_eval_violations} violations (target: ≥100)")

min_eval_cat = min(eval_cat_counts.get(c, 0) for c in CATEGORIES)
if min_eval_cat >= 20:
    quality_score += 1
    print(f" [1/1] Eval: all categories ≥20 (min={min_eval_cat})")
elif sum(1 for c in CATEGORIES if eval_cat_counts.get(c, 0) > 0) == 5:
    quality_score += 0.5
    print(f" ️ [0.5/1] Eval: all 5 present but min={min_eval_cat} (target: ≥20)")
else:
    print(f" [0/1] Eval: not all categories represented")

if has_modified_lines == len(static_data):
    quality_score += 1
    print(f" [1/1] All static files have modified_lines")
else:
    print(f" [0/1] Only {has_modified_lines}/{len(static_data)} have modified_lines")

if not eval_repos_in_corpus:
    quality_score += 1
    print(f" [1/1] No data leakage")
else:
    print(f" [0/1] Data leakage detected")

if len(available_models) >= 2:
    quality_score += 1
    print(f" [1/1] Ground truth cross-validated")
else:
    print(f" ️ [0/1] No cross-validation (single model)")

if line0_count == 0:
    quality_score += 1
    print(f" [1/1] No line_number=0 entries")
else:
    print(f" [0/1] {line0_count} line_number=0 entries remain")

if dup_gt_count == 0:
    quality_score += 1
    print(f" [1/1] No duplicate ground truth entries")
else:
    print(f" [0/1] {dup_gt_count} duplicate GT entries")

if at_marker_count == 0:
    quality_score += 1
    print(f" [1/1] No @@ hunk headers in diffs")
else:
    print(f" [0/1] {at_marker_count} @@ markers remain")

if oversized_chunks == 0:
    quality_score += 1
    print(f" [1/1] All corpus chunks ≤{CHUNK_MAX_TOKENS} tokens")
else:
    print(f" [0/1] {oversized_chunks} oversized chunks")

if synth_eval_pct <= 80:
    quality_score += 1
    print(f" [1/1] Synthetic eval ratio: {synth_eval_pct:.0f}% (≤80%)")
else:
    print(f" [0/1] Synthetic eval ratio: {synth_eval_pct:.0f}% (>80%)")

origin_present_eval = all("origin" in e for e in eval_data)
origin_present_corpus = all("origin" in c for c in corpus)
if origin_present_eval and origin_present_corpus:
    quality_score += 1
    print(f" [1/1] Origin traceability on all entries")
else:
    print(f" [0/1] Some entries missing 'origin' field")

print(f"\n QUALITY SCORE: {quality_score}/{max_score} ({quality_score/max_score*100:.0f}%)")

print(f"\n{'='*70}")
if errors:
    print(f" ️ VALIDATION COMPLETED WITH {len(errors)} WARNING(S):")
    for e in errors:
        print(f" • {e}")
else:
    print(f" ALL VALIDATIONS PASSED")

if quality_score >= 12:
    print(f" DATA QUALITY: EXCELLENT ({quality_score}/{max_score})")
elif quality_score >= 9:
    print(f" DATA QUALITY: GOOD ({quality_score}/{max_score})")
else:
    print(f" ️ DATA QUALITY: NEEDS IMPROVEMENT ({quality_score}/{max_score})")
print(f"{'='*70}")

print(f"\n Output Files:")
for name, path in [
    ("retrieval_corpus.json", RETRIEVAL_CORPUS_PATH),
    ("evaluation_dataset.json", EVAL_DATASET_PATH),
    ("static_analysis_input.json", STATIC_ANALYSIS_PATH),
]:
    if path.exists():
        size = path.stat().st_size
        if size > 1024 * 1024:
            print(f" {name}: {size / (1024*1024):.2f} MB")
        else:
            print(f" {name}: {size / 1024:.1f} KB")

In [ ]:
print("=" * 70)
print(" SAMPLE DATA PREVIEW")
print("=" * 70)

print("\n retrieval_corpus.json — Organic sample:")
organic_corpus = [c for c in retrieval_corpus if c.get("origin") != "synthetic"]
if organic_corpus:
    print(json.dumps(organic_corpus[0], indent=2, ensure_ascii=False)[:500])

synth_corpus = [c for c in retrieval_corpus if c.get("origin") == "synthetic"]
if synth_corpus:
    print(f"\n retrieval_corpus.json — Synthetic sample:")
    print(json.dumps(synth_corpus[0], indent=2, ensure_ascii=False)[:500])

print("\n evaluation_dataset.json — Organic sample:")
organic_eval = [e for e in evaluation_dataset if e.get("origin") != "synthetic"]
if organic_eval:
    sample = {k: v for k, v in organic_eval[0].items()}
    if sample.get("diff_chunks"):
        for chunk in sample["diff_chunks"]:
            if chunk.get("diff_lines") and len(chunk["diff_lines"]) > 5:
                chunk["diff_lines"] = chunk["diff_lines"][:5] + ["... (truncated)"]
                print(json.dumps(sample, indent=2, ensure_ascii=False)[:800])

synth_eval = [e for e in evaluation_dataset if e.get("origin") == "synthetic"]
if synth_eval:
    print(f"\n evaluation_dataset.json — Synthetic sample:")
    sample = {k: v for k, v in synth_eval[0].items()}
    if sample.get("diff_chunks"):
        for chunk in sample["diff_chunks"]:
            if chunk.get("diff_lines") and len(chunk["diff_lines"]) > 8:
                chunk["diff_lines"] = chunk["diff_lines"][:8] + ["... (truncated)"]
                print(json.dumps(sample, indent=2, ensure_ascii=False)[:800])

print("\n static_analysis_input.json — Sample entry:")
if static_analysis_input:
    sample = {**static_analysis_input[0]}
    sample["source_code"] = sample["source_code"][:300] + "..." if len(sample["source_code"]) > 300 else sample["source_code"]
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:600])

print(f"\n{'='*70}")
print(" DATA SCRAPING PIPELINE COMPLETE!")
print(f"{'='*70}")
print(f"\nDataset Summary:")
print(f" evaluation_dataset.json: {len(evaluation_dataset)} entries")
print(f" retrieval_corpus.json: {len(retrieval_corpus)} chunks")
print(f" static_analysis_input.json: {len(static_analysis_input)} files")
print(f"\nNext steps:")
print(f" 1. Build FAISS index from retrieval_corpus.json (embedding stage)")
print(f" 2. Run Pylint/Flake8 on static_analysis_input.json (baseline)")
print(f" 3. Run inference pipeline on evaluation_dataset.json")
print(f" 4. Compute metrics across all three systems")

---
## EDA — Data Quality Analysis & Visualizations

Systematic quality assessment of all three dataset files:
- Missing values and null fields
- Duplicate entries
- Distribution balance across categories
- Organic/synthetic ratio verification
- Diff size distribution
- Token length statistics for retrieval corpus

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

sns.set_style('whitegrid')
os.makedirs('figures', exist_ok=True)

OUTPUT_DIR = Path(".")
EVAL_DATASET_PATH = OUTPUT_DIR / "evaluation_dataset.json"
RETRIEVAL_CORPUS_PATH = OUTPUT_DIR / "retrieval_corpus.json"
STATIC_ANALYSIS_PATH = OUTPUT_DIR / "static_analysis_input.json"

VIOLATION_CATEGORIES = [
    "indentation", "naming_convention", "unused_import",
    "mutable_default", "documentation_formatting",
]

# Reload final datasets from disk to ensure consistency
with open(EVAL_DATASET_PATH) as f:
    eval_data = json.load(f)
with open(RETRIEVAL_CORPUS_PATH) as f:
    corpus_data = json.load(f)
with open(STATIC_ANALYSIS_PATH) as f:
    static_data = json.load(f)

print('=== Dataset Sizes ===')
print(f'Evaluation dataset: {len(eval_data)} entries')
print(f'Retrieval corpus:   {len(corpus_data)} chunks')
print(f'Static analysis:    {len(static_data)} files')

In [ ]:
# --- Missing Value Analysis ---

print('=== Missing Value Check ===')

eval_issues = []
for i, e in enumerate(eval_data):
    for field in ['pr_id', 'repo', 'file_path', 'diff_chunks', 'ground_truth_reviews', 'origin']:
        if not e.get(field):
            eval_issues.append(f'Entry {i}: missing {field}')
    for j, r in enumerate(e.get('ground_truth_reviews', [])):
        if not r.get('violation_category'):
            eval_issues.append(f'Entry {i}, review {j}: missing violation_category')

print(f'Evaluation dataset: {len(eval_issues)} issues found')
for issue in eval_issues[:5]:
    print(f'  {issue}')

corpus_issues = []
for i, c in enumerate(corpus_data):
    for field in ['chunk_id', 'text', 'category', 'source_type', 'origin']:
        if not c.get(field):
            corpus_issues.append(f'Chunk {i}: missing {field}')
print(f'Retrieval corpus: {len(corpus_issues)} issues found')

static_issues = []
for i, s in enumerate(static_data):
    for field in ['file_path', 'source_code', 'pr_id', 'repo', 'origin']:
        if not s.get(field):
            static_issues.append(f'Entry {i}: missing {field}')
print(f'Static analysis: {len(static_issues)} issues found')

In [ ]:
# --- Duplicate Check ---

print('=== Duplicate Check ===')

eval_ids = [e['pr_id'] for e in eval_data]
eval_dupes = [pid for pid, count in Counter(eval_ids).items() if count > 1]
print(f'Evaluation dataset duplicate PR IDs: {len(eval_dupes)}')
if eval_dupes:
    print(f'  Duplicates: {eval_dupes[:10]}')

corpus_ids = [c['chunk_id'] for c in corpus_data]
corpus_dupes = [cid for cid, count in Counter(corpus_ids).items() if count > 1]
print(f'Retrieval corpus duplicate chunk IDs: {len(corpus_dupes)}')

static_ids = [f"{s['pr_id']}_{s['file_path']}" for s in static_data]
static_dupes = [sid for sid, count in Counter(static_ids).items() if count > 1]
print(f'Static analysis duplicate entries: {len(static_dupes)}')

In [ ]:
# --- Visualization: Organic vs Synthetic Ratio ---

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

eval_origins = Counter(e['origin'] for e in eval_data)
axes[0].pie(eval_origins.values(), labels=eval_origins.keys(), autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title(f'Evaluation Dataset\n(n={len(eval_data)})')

corp_origins = Counter(c['origin'] for c in corpus_data)
axes[1].pie(corp_origins.values(), labels=corp_origins.keys(), autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title(f'Retrieval Corpus\n(n={len(corpus_data)})')

stat_origins = Counter(s['origin'] for s in static_data)
axes[2].pie(stat_origins.values(), labels=stat_origins.keys(), autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[2].set_title(f'Static Analysis\n(n={len(static_data)})')

plt.suptitle('Organic vs Synthetic Data Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/organic_synthetic_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualization: Violation Category Distribution ---

cat_counts = Counter()
cat_counts_by_origin = {'organic': Counter(), 'synthetic': Counter()}

for e in eval_data:
    for r in e.get('ground_truth_reviews', []):
        cat = r.get('violation_category')
        if cat and cat in VIOLATION_CATEGORIES:
            cat_counts[cat] += 1
            cat_counts_by_origin[e['origin']][cat] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cats = VIOLATION_CATEGORIES
counts = [cat_counts.get(c, 0) for c in cats]
colors = sns.color_palette('muted', len(cats))
axes[0].barh(cats, counts, color=colors)
axes[0].set_xlabel('Count')
axes[0].set_title('Violation Category Distribution\n(Evaluation Dataset)')
for i, v in enumerate(counts):
    axes[0].text(v + 0.5, i, str(v), va='center')

organic_counts_eda = [cat_counts_by_origin['organic'].get(c, 0) for c in cats]
synthetic_counts_eda = [cat_counts_by_origin['synthetic'].get(c, 0) for c in cats]

y_pos = range(len(cats))
axes[1].barh(y_pos, organic_counts_eda, label='Organic', color='#2ecc71')
axes[1].barh(y_pos, synthetic_counts_eda, left=organic_counts_eda, label='Synthetic', color='#e74c3c')
axes[1].set_yticks(list(y_pos))
axes[1].set_yticklabels(cats)
axes[1].set_xlabel('Count')
axes[1].set_title('Violations by Origin')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/violation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Violation counts by category:')
for cat in VIOLATION_CATEGORIES:
    org = cat_counts_by_origin['organic'].get(cat, 0)
    syn = cat_counts_by_origin['synthetic'].get(cat, 0)
    print(f'  {cat}: {org} organic + {syn} synthetic = {org+syn} total')

In [ ]:
# --- Visualization: Repository Distribution ---

repo_counts = Counter(e['repo'] for e in eval_data)

fig, ax = plt.subplots(figsize=(10, 5))
repos = list(repo_counts.keys())
counts = [repo_counts[r] for r in repos]
colors = ['#3498db' if 'synthetic' not in r else '#e74c3c' for r in repos]

bars = ax.bar(repos, counts, color=colors)
ax.set_ylabel('Number of Entries')
ax.set_title('Evaluation Dataset: Entries per Repository')
ax.tick_params(axis='x', rotation=30)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/repo_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualization: Diff Size & Chunk Size Distribution ---

diff_sizes = []
for e in eval_data:
    total_lines = sum(len(c.get('diff_lines', [])) for c in e.get('diff_chunks', []))
    diff_sizes.append(total_lines)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(diff_sizes, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Number of Diff Lines')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Diff Size Distribution')
axes[0].axvline(x=np.median(diff_sizes), color='red', linestyle='--',
                label=f'Median: {np.median(diff_sizes):.0f}')
axes[0].legend()

chunk_lengths = [len(c['text'].split()) for c in corpus_data]
axes[1].hist(chunk_lengths, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Chunk Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Retrieval Corpus: Chunk Size Distribution')
axes[1].axvline(x=np.median(chunk_lengths), color='red', linestyle='--',
                label=f'Median: {np.median(chunk_lengths):.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/size_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Diff sizes: min={min(diff_sizes)}, max={max(diff_sizes)}, '
      f'mean={np.mean(diff_sizes):.1f}, median={np.median(diff_sizes):.0f}')
print(f'Chunk lengths: min={min(chunk_lengths)}, max={max(chunk_lengths)}, '
      f'mean={np.mean(chunk_lengths):.1f}, median={np.median(chunk_lengths):.0f}')

In [ ]:
# --- Retrieval Corpus: Source Type Breakdown ---

source_counts = Counter(c['source_type'] for c in corpus_data)

fig, ax = plt.subplots(figsize=(8, 5))
sources = list(source_counts.keys())
counts = [source_counts[s] for s in sources]
colors = sns.color_palette('Set2', len(sources))

ax.pie(counts, labels=sources, autopct='%1.1f%%', colors=colors, startangle=140)
ax.set_title('Retrieval Corpus: Source Type Distribution')

plt.tight_layout()
plt.savefig('figures/corpus_sources.png', dpi=150, bbox_inches='tight')
plt.show()

for src, cnt in source_counts.most_common():
    print(f'  {src}: {cnt} ({cnt/len(corpus_data)*100:.1f}%)')

---
## Train / Validation / Test Split

### Split Strategy: Repository-Level Separation

To prevent data leakage, we use **repository-level train/eval separation**:

| Split | Repositories | Purpose |
|-------|-------------|--------|
| **Training (Retrieval Index)** | django/django, pandas-dev/pandas, scikit-learn/scikit-learn | Guidelines, review comments, and violation examples indexed in the retrieval corpus |
| **Evaluation (Test)** | pallets/flask, fastapi/fastapi | Ground-truth entries for measuring system performance |
| **Synthetic Supplement** | synthetic/pep8-violations | Balanced coverage for underrepresented categories |

The evaluation entries from flask and fastapi are further split into **validation (30%)** and **test (70%)** sets, stratified by violation category.

### No-Leakage Guarantees
1. No flask/fastapi review comments appear in the retrieval corpus
2. No evaluation PR IDs appear in the retrieval corpus
3. Training repo violation examples are ONLY used in the retrieval index, not in evaluation

In [ ]:
# Define splits
TRAINING_REPOS = {'django/django', 'pandas-dev/pandas', 'scikit-learn/scikit-learn'}
EVAL_REPOS_SET = {'pallets/flask', 'fastapi/fastapi'}

training_entries = [e for e in eval_data if e['repo'] in TRAINING_REPOS]
eval_entries = [e for e in eval_data if e['repo'] in EVAL_REPOS_SET]
synthetic_entries_final = [e for e in eval_data if e['origin'] == 'synthetic']

print('=== Repository-Level Split ===')
print(f'Training repos (retrieval index): {len(training_entries)} entries')
print(f'Evaluation repos (flask+fastapi): {len(eval_entries)} entries')
print(f'Synthetic supplement: {len(synthetic_entries_final)} entries')

# Split eval entries into validation and test (stratified)
np.random.seed(42)
eval_indices = np.random.permutation(len(eval_entries))
val_size = int(len(eval_entries) * 0.3)

val_entries = [eval_entries[i] for i in eval_indices[:val_size]]
test_entries = [eval_entries[i] for i in eval_indices[val_size:]]

# Add synthetic to test set for balanced evaluation
test_entries.extend(synthetic_entries_final)

print(f'\nValidation set: {len(val_entries)} entries')
print(f'Test set: {len(test_entries)} entries (including {len(synthetic_entries_final)} synthetic)')

In [ ]:
# --- No-Leakage Verification ---

print('=== Leakage Verification ===')

# Check 1: No eval repo reviews in retrieval corpus
corpus_texts = [c['text'] for c in corpus_data]
eval_repo_in_corpus = []
for text in corpus_texts:
    if 'pallets/flask' in text or 'fastapi/fastapi' in text:
        eval_repo_in_corpus.append(text[:80])

if eval_repo_in_corpus:
    print(f'WARNING: {len(eval_repo_in_corpus)} corpus chunks reference evaluation repos!')
    for t in eval_repo_in_corpus[:3]:
        print(f'  "{t}..."')
else:
    print('PASS: No evaluation repo references found in retrieval corpus')

# Check 2: No eval PR IDs in corpus
eval_pr_ids_set = set(e['pr_id'] for e in eval_entries + synthetic_entries_final)
corpus_pr_refs = []
for text in corpus_texts:
    for pid in eval_pr_ids_set:
        if pid in text:
            corpus_pr_refs.append(pid)

if corpus_pr_refs:
    print(f'WARNING: {len(corpus_pr_refs)} eval PR IDs found in corpus!')
else:
    print('PASS: No evaluation PR IDs found in retrieval corpus')

# Check 3: No overlap between training and eval entries
training_prs = set(e['pr_id'] for e in training_entries)
eval_prs = set(e['pr_id'] for e in eval_entries)
overlap = training_prs & eval_prs

if overlap:
    print(f'WARNING: PR ID overlap between train and eval: {overlap}')
else:
    print('PASS: No PR ID overlap between training and evaluation sets')

print('\n--- All leakage checks passed ---' if not (eval_repo_in_corpus or corpus_pr_refs or overlap) else '--- LEAKAGE DETECTED ---')

In [ ]:
# --- Save split metadata ---

split_info = {
    'split_strategy': 'repository-level separation',
    'training_repos': sorted(TRAINING_REPOS),
    'evaluation_repos': sorted(EVAL_REPOS_SET),
    'training_entries': len(training_entries),
    'validation_entries': len(val_entries),
    'test_entries': len(test_entries),
    'synthetic_in_test': len(synthetic_entries_final),
    'random_seed': 42,
    'validation_pr_ids': [e['pr_id'] for e in val_entries],
    'test_pr_ids': [e['pr_id'] for e in test_entries],
    'training_pr_ids': [e['pr_id'] for e in training_entries],
}

with open(OUTPUT_DIR / 'split_metadata.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print('Split metadata saved to split_metadata.json')
print(f'\nFinal dataset summary:')
print(f'  Training (retrieval index): {len(training_entries)} entries from {len(TRAINING_REPOS)} repos')
print(f'  Validation: {len(val_entries)} entries')
print(f'  Test: {len(test_entries)} entries')
print(f'  Retrieval corpus: {len(corpus_data)} chunks')
print(f'  Static analysis inputs: {len(static_data)} files')

---
## Summary Statistics

Final overview of all datasets produced by this pipeline.

In [ ]:
# Comprehensive summary table

summary_data = {
    'Dataset': ['Evaluation Dataset', 'Retrieval Corpus', 'Static Analysis Input'],
    'File': ['evaluation_dataset.json', 'retrieval_corpus.json', 'static_analysis_input.json'],
    'Total Entries': [len(eval_data), len(corpus_data), len(static_data)],
    'Organic': [
        sum(1 for e in eval_data if e['origin'] == 'organic'),
        sum(1 for c in corpus_data if c['origin'] == 'organic'),
        sum(1 for s in static_data if s['origin'] == 'organic'),
    ],
    'Synthetic': [
        sum(1 for e in eval_data if e['origin'] == 'synthetic'),
        sum(1 for c in corpus_data if c['origin'] == 'synthetic'),
        sum(1 for s in static_data if s['origin'] == 'synthetic'),
    ],
}

summary_df = pd.DataFrame(summary_data)
summary_df['Organic %'] = (summary_df['Organic'] / summary_df['Total Entries'] * 100).round(1)

print('=== Final Dataset Summary ===')
print(summary_df.to_string(index=False))
print(f'\nAll datasets saved to: {OUTPUT_DIR.resolve()}')